# Offline paired dev-120 reranking

Attach cached indexes and local model files. Disable Internet. No installation/download.

CPU is the default. Ask the user before any GPU run; expected T4 time: 5–10 minutes (unmeasured estimate), plus up to 5 minutes for startup/E5 encoding.

Edit input paths below, then run. Reports go to /kaggle/working. See RERANK_PLAN.md.

In [ ]:
from pathlib import Path
# Attach pre-existing indexes, E5 ONNX files and BGE weights as Kaggle inputs.
# Internet OFF. No pip install or model download is part of this notebook.
INDEX_ROOT = Path("/kaggle/input/cached-lecture-indexes")  # <video>/05_index
ENCODER_DIR = Path("/kaggle/input/local-e5-small")  # onnx/model.onnx + onnx/tokenizer.json
RERANKER_DIR = Path("/kaggle/input/local-bge-reranker-v2-m3")
QUERY_CACHE = None  # optional audited NPZ cache, matching the selected question IDs
OUTPUT_ROOT = Path("/kaggle/working/rerank-results")
RUN_NAME = "dev120-bge-rerank"
USE_GPU = False
# Set True only after the user explicitly approves this specific GPU run.
GPU_RUN_APPROVED = False


In [ ]:
files = {'pipeline/eval/offline_common.py': '"""Isolated access to the production notebook and backend functions."""\n\nimport os\n\nfor _name in set(os.environ) | {\n    "SUPABASE_DB_URL", "SUPABASE_URL", "SUPABASE_ANON_KEY",\n    "R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY",\n}:\n    if _name.startswith(("SUPABASE_", "R2_")):\n        os.environ[_name] = ""\nos.environ["HF_HUB_OFFLINE"] = "1"\nos.environ["TRANSFORMERS_OFFLINE"] = "1"\n\nimport ast\nimport importlib.util\nimport json\nimport sys\nimport types\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nROOT = Path(__file__).resolve().parents[1]\nREPO = ROOT.parent\nsys.path.insert(0, str(ROOT))\nfrom shared.source_paths import module_path\nEXPERIMENTS = ROOT / "runs" / "exp"\nVIDEOS = ["cs_0001", "D_kuulxKuxc", "hVU5Mw19V_c", "L25_V008", "L25_V011",\n          "L25_V037", "tHE1mC_n0-E", "v2YTz8ToEag", "y1CvB950VlU", "_eDGTHhyQ7M"]\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n",\n                    encoding="utf-8")\n\n\ndef experiment_dir(name, output_root=None):\n    base = Path(output_root).resolve() if output_root is not None else EXPERIMENTS.resolve()\n    path = (base / name).resolve()\n    if not path.is_relative_to(base) or path == base:\n        raise ValueError("Experiment name must stay inside its output root.")\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef load_notebook(name):\n    # Production embeds these source files into one namespace; they are not standalone modules.\n    module = types.ModuleType("offline_notebook_" + name)\n    extras = ["textclean", "adfilter"] if name == "alignment" else []\n    for filename in ["common", *extras, name]:\n        path = module_path(filename + ".py")\n        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), module.__dict__)\n    return module\n\n\ndef load_backend():\n    # Do not import app, storage, models, main, dotenv, supa or objects, even transiently.\n    name = "_lecture_eval_backend"\n    if name + ".features.search.retrieval" in sys.modules:\n        return sys.modules[name + ".features.search.retrieval"]\n    for package in (name, name + ".features", name + ".features.search", name + ".shared"):\n        module = types.ModuleType(package)\n        suffix = package.removeprefix(name).strip(".").split(".") if package != name else []\n        module.__path__ = [str(REPO / "backend" / "app" / Path(*suffix))]\n        sys.modules[package] = module\n    storage = types.ModuleType(name + ".shared.storage")\n    storage.DATA_DIR = EXPERIMENTS / "_unused_data"\n    storage.cloud = lambda: False\n    storage.load_video_meta = lambda video: None\n    sys.modules[storage.__name__] = storage\n    path = REPO / "backend" / "app" / "features" / "search" / "retrieval.py"\n    spec = importlib.util.spec_from_file_location(name + ".features.search.retrieval", path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = module\n    with patch.dict(sys.modules, {"app": sys.modules[name],\n                                 "app.shared": sys.modules[name + ".shared"],\n                                 "app.shared.storage": storage}):\n        spec.loader.exec_module(module)\n    return module\n\n\ndef production_config():\n    tree = ast.parse((ROOT / "builders/build_fast_pipeline.py").read_text(encoding="utf-8"))\n    config = next(ast.literal_eval(node.value) for node in tree.body\n                  if isinstance(node, ast.Assign)\n                  and any(isinstance(t, ast.Name) and t.id == "CONFIG" for t in node.targets))\n    values = {}\n    for node in ast.parse(config).body:\n        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):\n            try:\n                values[node.targets[0].id] = ast.literal_eval(node.value)\n            except ValueError:\n                pass\n    return {\n        "ad_suspect_overlap": values["AD_SUSPECT_OVERLAP"],\n        "speech_relevance": values["SPEECH_RELEVANCE"],\n        "slide_speech": values["SLIDE_SPEECH"] if values["AD_FILTER"] == "slide_speech" else None,\n        "embedding_model": values["EMBEDDING_MODEL"], "revision": values["EMBEDDING_REVISION"],\n        "dedupe_previous": values["OCR_DEDUPE_PREVIOUS"],\n        "conservative_ocr": values["CONSERVATIVE_OCR"],\n        "heading_titles": values["HEADING_TITLES"],\n        "asr_correction": values["ASR_CORRECTION"],\n        "failure_guard": values["FAILURE_GUARD"],\n        "speech_ad_filter": values["SPEECH_AD_FILTER"],\n        "chunk_tokens": 350, "overlap_tokens": 40, "batch_size": 32,\n        "speech_chunk_tokens": 180, "speech_maximum_seconds": 50.0,\n        "include_generated_descriptions": values["INCLUDE_GENERATED_DESCRIPTIONS"],\n        "generated_semantic_only": values["GENERATED_SEMANTIC_ONLY"],\n        "visual_state_timing": values.get("VISUAL_STATE_TIMING", True),\n    }\n\n\nclass TokenizerAdapter:\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def encode(self, text, add_special_tokens=False, **kwargs):\n        return self.tokenizer.encode(text, add_special_tokens=add_special_tokens).ids\n\n    def decode(self, tokens, skip_special_tokens=True):\n        text = self.tokenizer.decode(tokens, skip_special_tokens=skip_special_tokens)\n        # PreTrainedTokenizerBase.clean_up_tokenization, enabled by production XLM-R.\n        for before, after in [(" .", "."), (" ?", "?"), (" !", "!"), (" ,", ","),\n                              (" \' ", "\'"), (" n\'t", "n\'t"), (" \'m", "\'m"),\n                              (" \'s", "\'s"), (" \'ve", "\'ve"), (" \'re", "\'re")]:\n            text = text.replace(before, after)\n        return text\n\n\nclass ModelAdapter:\n    """Only the SentenceTransformer interface consumed by run_indexing, backed by real ONNX."""\n\n    max_seq_length = 512\n\n    def __init__(self, encoder, chunk_references=()):\n        from tokenizers import Tokenizer\n        self.encoder = encoder\n        # Chunk splitting must not inherit the query encoder\'s 512-token truncation.\n        tokenizer = Tokenizer.from_str(encoder.tokenizer.to_str())\n        tokenizer.no_truncation()\n        self.tokenizer = TokenizerAdapter(tokenizer)\n        self.chunk_vectors = {}\n        if chunk_references:\n            import numpy as np\n            import hashlib\n            for folder in reversed(chunk_references):\n                folder = Path(folder)\n                manifest = read_json(folder / "manifest.json")\n                for name in ("chunks.jsonl", "embeddings.npy"):\n                    if hashlib.sha256((folder / name).read_bytes()).hexdigest() != manifest["files"][name]:\n                        raise ValueError("Chunk reference checksum mismatch")\n                chunks = [json.loads(line) for line in (folder / "chunks.jsonl").read_text(encoding="utf-8").splitlines()]\n                vectors = np.load(folder / "embeddings.npy", allow_pickle=False)\n                if len(vectors) != len(chunks) or vectors.ndim != 2 or not np.isfinite(vectors).all():\n                    raise ValueError("Chunk reference vector count/finiteness mismatch")\n                for chunk, vector in zip(chunks, vectors):\n                    self.chunk_vectors[(chunk["chunk_id"], chunk["embedding_text"])] = vector.copy()\n\n    def _first_module(self):\n        return types.SimpleNamespace(auto_model=types.SimpleNamespace(\n            config=types.SimpleNamespace(_commit_hash=load_backend().E5_REVISION)))\n\n    def encode(self, texts, batch_size=32, **kwargs):\n        import numpy as np\n        # embedding_text already contains exactly one production passage prefix.\n        return np.concatenate([self.encoder.encode_many(texts[i:i + batch_size], prefix="")\n                               for i in range(0, len(texts), batch_size)])\n\n\n    def encode_chunks(self, chunks, batch_size=32, **kwargs):\n        """Preserve occurrence-specific rounding, so timing-only ablations keep ranks."""\n        import numpy as np\n        keys = [(c["chunk_id"], c["embedding_text"]) for c in chunks]\n        missing = [i for i, key in enumerate(keys) if key not in self.chunk_vectors]\n        vectors = self.encode([chunks[i]["embedding_text"] for i in missing], batch_size=batch_size) if missing else []\n        generated = dict(zip(missing, vectors))\n        return np.stack([self.chunk_vectors[key] if key in self.chunk_vectors else generated[i]\n                         for i, key in enumerate(keys)])\n\n\ndef encode_batches(encoder, texts, prefix, batch_size=32):\n    import numpy as np\n    return np.concatenate([encoder.encode_many(texts[i:i + batch_size], prefix=prefix)\n                           for i in range(0, len(texts), batch_size)])\n', 'pipeline/eval/eval_retrieval.py': '"""Measure backend retrieval ablations and the actual text returned by build_context."""\n\nimport os\n\nfor _name in set(os.environ) | {"SUPABASE_DB_URL", "SUPABASE_URL", "SUPABASE_ANON_KEY",\n                               "R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"}:\n    if _name.startswith(("SUPABASE_", "R2_")):\n        os.environ[_name] = ""\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport re\nimport sys\nfrom collections import defaultdict\nfrom datetime import datetime, timezone\nfrom difflib import SequenceMatcher\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom offline_common import (ROOT, VIDEOS, encode_batches, experiment_dir, load_backend,\n                            read_json, write_json)\n\nAD_PHRASES = ("tuyển sinh", "học phí", "học bổng", "đăng ký kênh", "ủng hộ kênh",\n              "kênh để ủng hộ", "subscribe", "hotline", "khuyến mãi", "siu.edu.vn",\n              "trường quốc tế á châu", "saigon international university")\n# Conservative catalogue: explicit contextual regressions documented in audit 3.\nBAD_CORRECTIONS = {("trong kinh", "chọn kinh"), ("trọng một", "chọn một"),\n                   ("chân không", "trạng không"), ("học chương", "học trường")}\nTYPES = {"speech", "slide", "both", "numeric", "term", "code", "visual", "unanswerable"}\n\n\ndef overlap(start, end, spans):\n    return any(start < b and end > a for a, b in spans)\n\n\ndef folded(text):\n    return load_backend().normalize_search(text)\n\n\ndef tokens(text):\n    return re.findall(r"[^\\W_]+", folded(text))\n\n\ndef term_match(term, text, threshold=0.88):\n    needle, haystack = tokens(term), tokens(text)\n    if not needle:\n        return False\n    wanted = " ".join(needle)\n    for i in range(len(haystack) - len(needle) + 1):\n        candidate = " ".join(haystack[i:i + len(needle)])\n        if candidate == wanted:\n            return True\n        # Numbers and short identifiers must match exactly (1.5 must never match 4.5).\n        if not any(c.isdigit() for c in wanted) and len(wanted) >= 5:\n            if SequenceMatcher(None, wanted, candidate, autojunk=False).ratio() >= threshold:\n                return True\n    return False\n\n\ndef repetition_runs(text, minimum=8):\n    words = tokens(text)\n    runs, i = [], 0\n    while i < len(words):\n        best = None\n        for width in range(1, 9):\n            phrase = words[i:i + width]\n            if len(phrase) < width or not any(any(c.isalpha() for c in w) for w in phrase):\n                continue\n            end = i + width\n            while words[end:end + width] == phrase:\n                end += width\n            count = (end - i) // width\n            if count >= minimum and (best is None or end > best["end_token"]):\n                best = {"phrase": " ".join(phrase), "repeats": count,\n                        "start_token": i, "end_token": end}\n        if best:\n            runs.append(best)\n            i = best["end_token"]\n        else:\n            i += 1\n    return runs\n\n\ndef context_trace(found, index, question, context):\n    """Track retained words/chunks through production\'s character slicing, without changing it."""\n    if "context_evidence" in found:\n        trace = evidence_context_trace(found, index, question, context)\n        if found.get("context_legacy"):\n            legacy = context_trace({"slides": found["slides"], "bounded_legacy": True}, index, question, found["context_legacy"])\n            trace["recalled"] = trace["recalled"] or legacy["recalled"]\n            trace["legacy"] = legacy\n        return trace\n    spans, entries = question["gold_spans"], []\n    word_file = index.folder / "lecture" / "words.json"\n    words = {w["word_id"]: w for w in read_json(word_file)} if word_file.exists() else {}\n    for slide in found["slides"]:\n        sid = slide["segment_id"]\n        segment = index.by_segment.get(sid, {})\n        full_speech, speech = slide["speech_text"], slide["speech_text"][:2500]\n        ocr = slide["ocr_text"][:1800]\n        if found.get("bounded_legacy"):\n            start = context.find(f"Slide {sid.split(\':\')[-1]} · ")\n            end = context.find("\\n\\n[", start) if start >= 0 else 0\n            block = context[start:end if end >= 0 else len(context)] if start >= 0 else ""\n            speech_start = block.find("Lời giảng:\\n")\n            speech = speech[:len(block[speech_start + len("Lời giảng:\\n"):])] if speech_start >= 0 else ""\n            ocr_start = block.find("Chữ trên slide (OCR, có thể sai số/dấu — đối chiếu ảnh nếu có):\\n")\n            if ocr_start < 0:\n                ocr = ""\n            else:\n                body = block[ocr_start:].split("\\n", 1)[1]\n                ocr = ocr[:len(body)]\n        selected, offset, missing = [], 0, []\n        for wid in segment.get("speech", {}).get("word_ids", []):\n            word = words.get(wid)\n            if word is None:\n                missing.append(wid)\n                continue\n            text = word["text"].strip()\n            pos = full_speech.find(text, offset)\n            if pos < 0:\n                missing.append(wid)\n                continue\n            offset = pos + len(text)\n            if offset <= len(speech):\n                selected.append(word)\n        retained_chunks = []\n        for c in index.chunks:\n            if c["segment_id"] != sid:\n                continue\n            body = speech if c["source_type"] == "speech" else ocr\n            if c["source_type"] == "region":\n                body = "\\n".join(f[:600] for f in slide["figures"][:3])\n            # Require most of the chunk\'s words to survive; keep partial overlaps explicit.\n            ct, bt = tokens(c["text"]), tokens(body)\n            share = SequenceMatcher(None, ct, bt, autojunk=False).find_longest_match(\n                0, len(ct), 0, len(bt)).size / max(1, len(ct))\n            if share > 0:\n                retained_chunks.append({"chunk_id": c["chunk_id"], "source_type": c["source_type"],\n                                        "start_sec": c["start_sec"], "end_sec": c["end_sec"],\n                                        "retained_token_share": round(share, 4)})\n        timed_words = [w for w in selected if overlap(w["start_sec"], w["end_sec"], spans)]\n        if words and segment.get("speech", {}).get("word_ids"):\n            speech_evidence = " ".join(w["text"] for w in timed_words)\n            speech_time = bool(timed_words)\n        else:\n            speech_time = any(c["source_type"] == "speech" and c["retained_token_share"] >= 0.5\n                              and overlap(c["start_sec"], c["end_sec"], spans) for c in retained_chunks)\n            speech_evidence = speech if speech_time else ""\n        visual_text = "\\n".join([slide["title"], ocr])\n        ocr_time = bool(visual_text.strip()) and overlap(slide["start_sec"], slide["end_sec"], spans)\n        figure_text = "\\n".join(f[:600] for f in slide["figures"][:3])\n        terms = question["key_terms"]\n        def checked(text, timing):\n            matched = [term_match(t, text) and term_match(t, context) for t in terms]\n            return {"time_overlap": timing, "term_matches": matched,\n                    "recalled": timing and all(matched)}\n        source = question.get("evidence_source", "either")\n        checks = {"speech": checked(speech_evidence, speech_time),\n                  "ocr": checked(visual_text, ocr_time),\n                  "region": checked(figure_text, bool(figure_text) and ocr_time)}\n        eligible = [checks[source]] if source in checks else list(checks.values())\n        entries.append({"segment_id": sid, "speech_chars_included": len(speech),\n                        "speech_chars_omitted": max(0, len(full_speech) - len(speech)),\n                        "ocr_chars_included": len(ocr), "included_word_ids": [w["word_id"] for w in selected],\n                        "gold_overlap_word_ids": [w["word_id"] for w in timed_words],\n                        "unmapped_word_ids": missing, "retained_chunks": retained_chunks,\n                        "checks": checks, "recalled": any(c["recalled"] for c in eligible)})\n    return {"recalled": any(e["recalled"] for e in entries), "slides": entries,\n            "definition": "Gold time overlap AND all annotated key terms in the retained evidence and build_context text."}\n\n\ndef retained_words(chunk, words):\n    candidates = [w for w in words if w["start_sec"] < chunk["end_sec"] and w["end_sec"] > chunk["start_sec"]]\n    word_tokens, owners, counts = [], [], {}\n    for word in candidates:\n        pieces = tokens(word["text"])\n        word_tokens.extend(pieces)\n        owners.extend([word["word_id"]] * len(pieces))\n        counts[word["word_id"]] = len(pieces)\n    included = defaultdict(int)\n    for match in SequenceMatcher(None, word_tokens, tokens(chunk["text"]), autojunk=False).get_matching_blocks():\n        for owner in owners[match.a:match.a + match.size]:\n            included[owner] += 1\n    return [w for w in candidates if counts[w["word_id"]] and included[w["word_id"]] == counts[w["word_id"]]]\n\n\ndef evidence_context_trace(found, index, question, context):\n    """Use only retained blocks; neighbors keep their own segment and word-span timing."""\n    word_file = index.folder / "lecture" / "words.json"\n    words = read_json(word_file) if word_file.exists() else []\n    entries = []\n    by_segment = defaultdict(list)\n    for chunk in found["context_evidence"]:\n        by_segment[chunk["segment_id"]].append(chunk)\n    for sid, chunks in by_segment.items():\n        speech = [c for c in chunks if c["source_type"] == "speech"]\n        selected = {w["word_id"]: w for c in speech for w in retained_words(c, words)}\n        selected = sorted(selected.values(), key=lambda w: (w["start_sec"], w["end_sec"]))\n        timed = [w for w in selected if overlap(w["start_sec"], w["end_sec"], question["gold_spans"])]\n        speech_text = " ".join(w["text"] for w in timed) if words else "\\n".join(\n            c["text"] for c in speech if overlap(c["start_sec"], c["end_sec"], question["gold_spans"]))\n        texts = {"speech": speech_text}\n        for kind in ("ocr", "region"):\n            texts[kind] = "\\n".join(c.get("title", "") + "\\n" + c["text"] for c in chunks\n                                    if c["source_type"] == kind and overlap(\n                                        c["start_sec"], c["end_sec"], question["gold_spans"]))\n        checks = {}\n        for kind, text in texts.items():\n            matched = [term_match(t, text) and term_match(t, context) for t in question["key_terms"]]\n            checks[kind] = {"time_overlap": bool(text), "term_matches": matched,\n                            "recalled": bool(text) and all(matched)}\n        source = question.get("evidence_source", "either")\n        eligible = [checks[source]] if source in checks else list(checks.values())\n        entries.append({"segment_id": sid, "included_word_ids": [w["word_id"] for w in selected],\n                        "gold_overlap_word_ids": [w["word_id"] for w in timed], "unmapped_word_ids": [],\n                        "retained_chunks": [{k: c[k] for k in ("chunk_id", "source_type", "start_sec", "end_sec",\n                                                                 "selection_reason")} for c in chunks],\n                        "checks": checks, "recalled": any(c["recalled"] for c in eligible)})\n    return {"recalled": any(e["recalled"] for e in entries), "slides": entries,\n            "definition": "Gold time overlap AND all key terms in actually retained evidence and context."}\n\n\ndef quality_lints(index):\n    chunks = index.chunks\n    promotions = [c["chunk_id"] for c in chunks\n                  if any(folded(p) in folded(c["text"]) for p in AD_PHRASES)]\n    ocr = [c for c in chunks if c["source_type"] == "ocr"]\n    groups = defaultdict(list)\n    for c in ocr:\n        groups[c["text"]].append(c["chunk_id"])\n    unique = [next(c for c in ocr if c["chunk_id"] == ids[0]) for ids in groups.values()]\n    near = []\n    for i, a in enumerate(unique):\n        at = tokens(a["text"])\n        ash = {tuple(at[j:j + 3]) for j in range(len(at) - 2)}\n        if len(at) < 16:\n            continue\n        for b in unique[i + 1:]:\n            bt = tokens(b["text"])\n            bsh = {tuple(bt[j:j + 3]) for j in range(len(bt) - 2)}\n            ratio = len(ash & bsh) / max(1, len(ash | bsh))\n            if len(bt) >= 16 and ratio >= 0.9:\n                near.append({"a": a["chunk_id"], "b": b["chunk_id"], "jaccard": round(ratio, 4)})\n    loops = []\n    for c in chunks:\n        runs = repetition_runs(c["text"])\n        if runs:\n            loops.append({"chunk_id": c["chunk_id"], "source_type": c["source_type"], "runs": runs})\n    folder = index.folder / "lecture"\n    corrections = read_json(folder / "asr_corrections.json") if (folder / "asr_corrections.json").exists() else []\n    words = read_json(folder / "words.json") if (folder / "words.json").exists() else []\n    changed = [w for w in words if "text_asr" in w and w["text_asr"] != w["text"]]\n    bad = [c for c in corrections if (c["heard"], c["fixed"]) in BAD_CORRECTIONS\n           and any(abs(w["start_sec"] - c["at_sec"]) < 3 for w in changed)]\n    empty = sum(not s.get("visual", {}).get("title_clean", s.get("visual", {}).get("title"))\n                for s in index.segments)\n    return {"chunks": len(chunks), "segments": len(index.segments), "ocr_chunks": len(ocr),\n            "promotional_chunks": len(promotions), "promotional_chunk_ids": promotions,\n            "exact_duplicate_ocr_excess": sum(len(ids) - 1 for ids in groups.values()),\n            "exact_duplicate_ocr_groups": [ids for ids in groups.values() if len(ids) > 1],\n            "near_duplicate_unique_ocr_pairs": len(near), "near_duplicate_pairs": near,\n            "ocr_loop_chunks": sum(c["source_type"] == "ocr" for c in loops),\n            "asr_loop_chunks": sum(c["source_type"] == "speech" for c in loops), "loops": loops,\n            "chunks_under_32_tokens": sum(c["encoded_tokens"] < 32 for c in chunks),\n            "empty_titles": empty, "empty_title_rate": empty / max(1, len(index.segments)),\n            "empty_chunk_title_rate": sum(not c.get("title") for c in chunks) / max(1, len(chunks)),\n            "asr_correction_events": len(corrections), "changed_asr_words": len(changed),\n            "known_bad_asr_corrections": len(bad), "known_bad_events": bad,\n            "known_bad_catalogue": sorted([list(x) for x in BAD_CORRECTIONS])}\n\n\ndef validate_questions(data):\n    seen = set()\n    for q in data["questions"]:\n        if q["id"] in seen or q["type"] not in TYPES or q["video"] not in VIDEOS:\n            raise ValueError(f"Invalid question: {q[\'id\']}")\n        seen.add(q["id"])\n        if q["type"] == "unanswerable":\n            if q["gold_spans"]:\n                raise ValueError("Unanswerable questions must have no gold spans")\n        elif not q["gold_spans"] or not q.get("key_terms") or not q["answer"]:\n            raise ValueError(f"Missing gold evidence: {q[\'id\']}")\n        for a, b in q["gold_spans"]:\n            if not (math.isfinite(a) and math.isfinite(b) and 0 <= a < b):\n                raise ValueError(f"Invalid span: {q[\'id\']}")\n\n\ndef aggregate(rows):\n    answered = [r for r in rows if r["type"] != "unanswerable"]\n    n = len(answered)\n    return {"answerable": n, "unanswerable": len(rows) - n,\n            "avg_context_chars": sum(len(r.get("context", "")) for r in answered) / n if n else None,\n            "max_context_chars": max((len(r.get("context", "")) for r in answered), default=0),\n            **{key: sum(r[key] for r in answered) / n if n else None\n               for key in ("hit1", "hit3", "hit5", "mrr", "context_recall")},\n            "unanswerable_top_scores": [{"id": r["id"], "semantic": r["best_semantic"],\n                                         "rrf": r["top_rrf"]}\n                                        for r in rows if r["type"] == "unanswerable"]}\n\n\ndef markdown(report):\n    lines = ["# Retrieval evaluation", "", f"Questions: {report[\'question_status\']}", "",\n             "Hit@5 uses the unchanged production maximum of three slides, so it equals Hit@3.", "",\n             "| Video / type | Answerable | Hit@1 | Hit@3 | Hit@5 | MRR | Context recall | Avg context chars |",\n             "|---|---:|---:|---:|---:|---:|---:|---:|"]\n    for label, metrics in [*report["per_video"].items(), ("OVERALL", report["overall"]),\n                           *[("type: " + k, v) for k, v in report["per_type"].items()]]:\n        if not metrics["answerable"]:\n            continue\n        values = [f"{metrics[k]:.3f}" for k in ("hit1", "hit3", "hit5", "mrr", "context_recall")]\n        lines.append(f"| {label} | {metrics[\'answerable\']} | " + " | ".join(values) + f" | {metrics[\'avg_context_chars\']:.0f} |")\n    lines += ["", "| Video | Chunks | Ads* | Exact OCR excess | Near OCR pairs* | OCR / ASR loop chunks* | <32 tokens | Empty titles | Bad corrections* |",\n              "|---|---:|---:|---:|---:|---:|---:|---:|---:|"]\n    for video, lint in report["quality"].items():\n        lines.append(f"| {video} | {lint[\'chunks\']} | {lint[\'promotional_chunks\']} | {lint[\'exact_duplicate_ocr_excess\']} | "\n                     f"{lint[\'near_duplicate_unique_ocr_pairs\']} | {lint[\'ocr_loop_chunks\']} / {lint[\'asr_loop_chunks\']} | "\n                     f"{lint[\'chunks_under_32_tokens\']} | {lint[\'empty_titles\']}/{lint[\'segments\']} | {lint[\'known_bad_asr_corrections\']} |")\n    lines += ["", "*Diagnostic rules, not verified corpus error rates. Near pairs: unique OCR texts, 3-token Jaccard ≥0.90 and ≥16 tokens; loops: ≥8 consecutive repeats of 1–8 tokens. Ads: fixed phrase catalogue. Bad corrections: four documented phrase pairs, checked against text_asr changes.",\n              "", "| Unanswerable ID | Top E5 cosine | Top fused RRF |", "|---|---:|---:|"]\n    for row in report["overall"]["unanswerable_top_scores"]:\n        lines.append(f"| {row[\'id\']} | {row[\'semantic\']:.4f} | {row[\'rrf\']:.6f} |")\n    return "\\n".join(lines) + "\\n"\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    group = parser.add_mutually_exclusive_group(required=True)\n    group.add_argument("--index-root", type=Path, help="Root containing <video>/05_index")\n    group.add_argument("--index-dir", type=Path, help="A single stage-05 index; use --videos VIDEO")\n    parser.add_argument("--videos", nargs="+", default=VIDEOS, choices=VIDEOS)\n    parser.add_argument("--questions", type=Path, default=ROOT / "eval/questions_dev.json")\n    parser.add_argument("--name", required=True)\n    parser.add_argument("--output-root", type=Path, help="Root for evaluation reports")\n    parser.add_argument("--features", nargs="*", choices=("ranked_context", "neighbors", "vietnamese_lexical",\n                                                        "phrases", "dedup", "diversity", "no_boost"),\n                        default=None, help="Omit for app defaults; empty list reproduces Phase 1.")\n    args = parser.parse_args()\n    if args.index_dir and len(args.videos) != 1:\n        parser.error("--index-dir requires exactly one --videos value")\n    data = read_json(args.questions)\n    validate_questions(data)\n    questions = [q for q in data["questions"] if q["video"] in args.videos]\n    output = experiment_dir(args.name, args.output_root)\n    folders = {v: args.index_dir or args.index_root / v / "05_index" for v in args.videos}\n    backend = load_backend()\n    options = backend.retrieval_options(args.features)\n    encoder = backend.OnnxEncoder()\n    vectors = encode_batches(encoder, [q["question"] for q in questions], "query: ")\n    report = {"question_status": data["status"], "questions_sha256": hashlib.sha256(args.questions.read_bytes()).hexdigest(),\n              "measured_utc": datetime.now(timezone.utc).isoformat(),\n              "harness_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()\n                                 for p in [Path(__file__), ROOT / "eval/offline_common.py"]},\n              "backend_sha256": hashlib.sha256(Path(backend.__file__).read_bytes()).hexdigest(),\n              "encoder": {"model": backend.E5_REPO, "revision": backend.E5_REVISION, "device": "CPU ONNX"},\n              "retrieval": {"max_slides": backend.MAX_SLIDES, "candidates": backend.CANDIDATES,\n                            "rrf_k": backend.RRF_K, "semantic_weight": backend.SEMANTIC_WEIGHT,\n                            "lexical_weight": backend.LEXICAL_WEIGHT, "source_boost": backend.SOURCE_BOOST,\n                            "speech_chars": 2500, "ocr_chars": 1800,\n                            "options": backend.asdict(options), "max_context_chars": backend.MAX_CONTEXT_CHARS},\n              "questions": [], "quality": {},\n              "index_inputs": {v: {"path": str(p.resolve()),\n                                   "chunks_sha256": hashlib.sha256((p / "chunks.jsonl").read_bytes()).hexdigest()}\n                               for v, p in folders.items()}}\n    backend.load_index.cache_clear()\n    with patch.object(backend, "index_dir", side_effect=lambda v: folders[v]):\n        for video in args.videos:\n            index = backend.load_index(video)\n            report["quality"][video] = quality_lints(index)\n        for question_index, (q, vector) in enumerate(zip(questions, vectors)):\n            fixed = type("EncodedQuery", (), {"encode": lambda self, text, v=vector: v})()\n            found = backend.search(q["video"], q["question"], encoder=fixed, options=options)\n            context = backend.build_context(found)\n            hits = [overlap(s["start_sec"], s["end_sec"], q["gold_spans"]) for s in found["slides"]]\n            row = {"id": q["id"], "video": q["video"], "type": q["type"], "question": q["question"],\n                   "best_semantic": found["best_semantic"],\n                   "top_rrf": found["slides"][0]["score"] if found["slides"] else 0,\n                   "slides": [{k: v for k, v in s.items() if k not in {"speech_text", "ocr_text", "figures"}}\n                              for s in found["slides"]], "context": context,\n                   "context_chars": len(context), "context_evidence": found.get("context_evidence", [])}\n            if q["type"] != "unanswerable":\n                trace = context_trace(found, backend.load_index(q["video"]), q, context)\n                row.update({"hit1": int(any(hits[:1])), "hit3": int(any(hits[:3])),\n                            "hit5": int(any(hits[:5])), "mrr": next((1 / (i + 1) for i, h in enumerate(hits) if h), 0),\n                            "context_recall": int(trace["recalled"]), "context_trace": trace})\n            report["questions"].append(row)\n            if question_index + 1 == len(questions) or questions[question_index + 1]["video"] != q["video"]:\n                print(q["video"], "evaluated", flush=True)\n    backend.load_index.cache_clear()\n    report["overall"] = aggregate(report["questions"])\n    report["per_video"] = {v: aggregate([r for r in report["questions"] if r["video"] == v]) for v in args.videos}\n    report["per_type"] = {t: aggregate([r for r in report["questions"] if r["type"] == t]) for t in sorted(TYPES)}\n    write_json(output / "retrieval.json", report)\n    (output / "retrieval.md").write_text(markdown(report), encoding="utf-8")\n    if "torch" in sys.modules:\n        raise RuntimeError("Unexpected torch import")\n    print(json.dumps(report["overall"], ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/eval/rerank_eval.py': '"""Offline paired dev-120 retrieval evaluation; never rebuilds indexes or downloads models.\n\nDo not launch a GPU run without explicit user approval. See notes/RERANK_PLAN.md.\n"""\n\nimport argparse\nimport hashlib\nimport importlib\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom time import perf_counter\nfrom unittest.mock import patch\n\nfrom offline_common import (ROOT, VIDEOS, encode_batches, experiment_dir, load_backend,\n                            read_json, write_json)\nfrom eval_retrieval import context_trace, overlap, validate_questions\n\nMETRICS = ("hit1", "hit3", "mrr3", "context_success")\n\n\ndef measure(backend, index, question, found):\n    context = backend.build_context(found)\n    row = {"slides": [{k: slide[k] for k in ("segment_id", "start_sec", "end_sec", "rank",\n                                           "matched_chunk_ids")} for slide in found["slides"]],\n           "best_semantic": found["best_semantic"], "best_rerank": found.get("best_rerank"),\n           "context": context, "context_chars": len(context),\n           "context_evidence": found.get("context_evidence", [])}\n    if question["type"] != "unanswerable":\n        hits = [overlap(s["start_sec"], s["end_sec"], question["gold_spans"])\n                for s in found["slides"][:3]]\n        trace = context_trace(found, index, question, context)\n        row.update(hit1=int(any(hits[:1])), hit3=int(any(hits)),\n                   mrr3=next((1 / (rank + 1) for rank, hit in enumerate(hits) if hit), 0.0),\n                   context_success=int(trace["recalled"]), context_trace=trace)\n    return row\n\n\ndef paired_summary(rows):\n    answerable = [r for r in rows if r["type"] != "unanswerable"]\n    n = len(answerable)\n    before = {key: sum(r["baseline"][key] for r in answerable) / n if n else None for key in METRICS}\n    after = {key: sum(r["rerank"][key] for r in answerable) / n if n else None for key in METRICS}\n    return {"answerable": n, "unanswerable": len(rows) - n,\n            "baseline": before, "rerank": after,\n            "paired_delta": {key: sum(r["delta"][key] for r in answerable) / n if n else None\n                             for key in METRICS},\n            "improved": {key: sum(r["delta"][key] > 0 for r in answerable) for key in METRICS},\n            "regressed": {key: sum(r["delta"][key] < 0 for r in answerable) for key in METRICS}}\n\n\ndef evaluate_pair(backend, folders, questions, vectors, *, scorer=None):\n    """Both arms use the same immutable indexes, E5 vectors and retrieval options."""\n    import numpy as np\n\n    vectors = np.asarray(vectors)\n    if vectors.ndim != 2 or len(vectors) != len(questions) or not np.isfinite(vectors).all():\n        raise ValueError("Expected one finite query vector per question")\n    options = backend.RetrievalOptions()\n    rows = []\n    backend.load_index.cache_clear()\n    try:\n        with patch.object(backend, "index_dir", side_effect=lambda video: folders[video]):\n            for question, vector in zip(questions, vectors):\n                index = backend.load_index(question["video"])\n                if index.model != backend.E5_REPO:\n                    raise ValueError("dev-120 evaluator requires existing multilingual-e5-small indexes")\n                if index.vectors.ndim != 2 or index.vectors.shape[1] != len(vector):\n                    raise ValueError("Query/index embedding dimensions do not match")\n                fixed = type("EncodedQuery", (), {"encode": lambda self, text, v=vector: v})()\n                row = {key: question[key] for key in ("id", "video", "type", "question")}\n                for label, enabled in (("baseline", False), ("rerank", True)):\n                    started = perf_counter()\n                    with patch.object(backend, "RERANK", enabled):\n                        found = backend.search(question["video"], question["question"], encoder=fixed,\n                                               options=options, rerank_scorer=scorer)\n                    row[label] = measure(backend, index, question, found)\n                    row[label]["elapsed_sec"] = perf_counter() - started\n                if question["type"] != "unanswerable":\n                    row["delta"] = {key: row["rerank"][key] - row["baseline"][key] for key in METRICS}\n                rows.append(row)\n    finally:\n        backend.load_index.cache_clear()\n    return {"questions": rows, "overall": paired_summary(rows),\n            "per_video": {video: paired_summary([r for r in rows if r["video"] == video])\n                          for video in sorted({r["video"] for r in rows})},\n            "per_type": {kind: paired_summary([r for r in rows if r["type"] == kind])\n                         for kind in sorted({r["type"] for r in rows})}}\n\n\ndef query_vectors(backend, questions, questions_hash, cache=None, encoder_dir=None):\n    """Optional audited NPZ cache, otherwise local-only ONNX E5 query encoding."""\n    import numpy as np\n\n    if cache is not None:\n        with np.load(cache, allow_pickle=False) as data:\n            if (data["question_ids"].tolist() != [q["id"] for q in questions]\n                    or str(data["model"].item()) != backend.E5_REPO\n                    or str(data["revision"].item()) != backend.E5_REVISION\n                    or str(data["questions_sha256"].item()) != questions_hash):\n                raise ValueError("Query cache IDs/model/revision/questions checksum mismatch")\n            return data["vectors"].copy()\n    from huggingface_hub import hf_hub_download\n\n    def local_file(repo, filename, **kwargs):\n        if encoder_dir is not None:\n            path = encoder_dir / filename\n            if not path.is_file():\n                raise FileNotFoundError(f"Missing local E5 asset: {path}")\n            return str(path)\n        return hf_hub_download(repo, filename, **kwargs, local_files_only=True)\n\n    with patch("huggingface_hub.hf_hub_download", side_effect=local_file):\n        encoder = backend.OnnxEncoder()\n    return encode_batches(encoder, [q["question"] for q in questions], "query: ")\n\n\ndef markdown(report):\n    lines = ["# Paired dev-120 reranking", "", f"Questions: {report[\'question_status\']}", "",\n             "Metrics exclude unanswerable questions. MRR@3 stops at the third selected slide.",\n             "Context success requires gold time overlap and all annotated terms in retained text",\n             "from the requested evidence source (the existing context_trace definition).", "",\n             "| Metric | Rerank off | Rerank on | Paired delta | Improved | Regressed |",\n             "|---|---:|---:|---:|---:|---:|"]\n    summary = report["overall"]\n    for key in METRICS:\n        if summary["baseline"][key] is not None:\n            lines.append(f"| {key} | {summary[\'baseline\'][key]:.4f} | {summary[\'rerank\'][key]:.4f} | "\n                         f"{summary[\'paired_delta\'][key]:+.4f} | {summary[\'improved\'][key]} | "\n                         f"{summary[\'regressed\'][key]} |")\n    lines += ["", f"Answerable: {summary[\'answerable\']}; unanswerable: {summary[\'unanswerable\']}.",\n              "Unanswerable rows retain best scores for later A8 calibration; no threshold was tuned.",\n              "JSON includes per-question paired deltas, context traces, per-video and per-type metrics.",\n              "Sigmoid relevance scores are uncalibrated; this run measures retrieval, not Q8 accuracy."]\n    return "\\n".join(lines) + "\\n"\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef main(*, scorer=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    group = parser.add_mutually_exclusive_group(required=True)\n    group.add_argument("--index-root", type=Path, help="Cached export root containing <video>/05_index")\n    group.add_argument("--index-dir", type=Path, help="One cached 05_index; requires one --videos value")\n    parser.add_argument("--videos", nargs="+", choices=VIDEOS, default=VIDEOS)\n    parser.add_argument("--questions", type=Path, default=ROOT / "eval/questions_dev.json")\n    parser.add_argument("--query-vectors", type=Path, help="Audited NPZ cache; see RERANK_PLAN.md")\n    parser.add_argument("--encoder-dir", type=Path, help="Local E5 snapshot with onnx assets")\n    parser.add_argument("--reranker-dir", type=Path, help="Already provisioned local BGE snapshot/export")\n    parser.add_argument("--name", required=True)\n    parser.add_argument("--output-root", type=Path)\n    args = parser.parse_args()\n    if args.index_dir and len(args.videos) != 1:\n        parser.error("--index-dir requires exactly one --videos value")\n    data = read_json(args.questions)\n    validate_questions(data)\n    if len(data["questions"]) != 120:\n        parser.error("Expected the full dev-120 question bank; do not use the held-out test set")\n    videos = list(dict.fromkeys(args.videos))\n    questions = [q for q in data["questions"] if q["video"] in videos]\n    folders = {v: (args.index_dir or args.index_root / v / "05_index").resolve() for v in videos}\n    inputs = {}\n    for video, folder in folders.items():\n        names = ["chunks.jsonl", "embeddings.npy", "lexical.sqlite3", "manifest.json", "lecture/segments.json"]\n        if (folder / "lecture/words.json").exists():\n            names.append("lecture/words.json")\n        inputs[video] = {"path": str(folder), "sha256": {name: sha256(folder / name) for name in names}}\n    backend = load_backend()\n    reranker = importlib.import_module(backend.__package__ + ".rerank")\n    if args.reranker_dir:\n        os.environ["LECTURE_RERANK_MODEL_DIR"] = str(args.reranker_dir.resolve())\n    questions_hash = sha256(args.questions)\n    vectors = query_vectors(backend, questions, questions_hash, args.query_vectors, args.encoder_dir)\n    report = evaluate_pair(backend, folders, questions, vectors, scorer=scorer)\n    report.update(question_status=data["status"], questions_sha256=questions_hash,\n                  measured_utc=datetime.now(timezone.utc).isoformat(), index_inputs=inputs,\n                  sources_sha256={str(p.relative_to(ROOT.parent)): sha256(p) for p in\n                                  (Path(__file__), ROOT / "eval/offline_common.py", ROOT / "eval/eval_retrieval.py",\n                                   Path(backend.__file__), Path(reranker.__file__))},\n                  retrieval={"options": backend.asdict(backend.RetrievalOptions()),\n                             "max_slides": backend.MAX_SLIDES, "rerank_candidates": backend.RERANK_CANDIDATES,\n                             "rrf_k": backend.RRF_K, "semantic_weight": backend.SEMANTIC_WEIGHT,\n                             "lexical_weight": backend.LEXICAL_WEIGHT},\n                  encoder={"model": backend.E5_REPO, "revision": backend.E5_REVISION,\n                           "query_cache_sha256": sha256(args.query_vectors) if args.query_vectors else None},\n                  reranker={"model": reranker.RERANK_REPO, "revision": reranker.RERANK_REVISION,\n                            "local_dir": os.environ.get("LECTURE_RERANK_MODEL_DIR"),\n                            "max_tokens": reranker.MAX_TOKENS, "batch_size": reranker.BATCH_SIZE,\n                            "scorer": getattr(scorer, "backend_name", "local-only CPU (ONNX preferred)"),\n                            "calibrated": False})\n    output = experiment_dir(args.name, args.output_root)\n    write_json(output / "rerank.json", report)\n    (output / "rerank.md").write_text(markdown(report), encoding="utf-8")\n    print(json.dumps(report["overall"], ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/eval/questions_dev.json': '{\n  "status": "auto-written by Codex, needs human review",\n  "version": 1,\n  "authoring": {\n    "method": "Codex visually inspected 71 selected stage-01 keyframes (saved as contact sheets) and read corresponding raw stage-03 word spans plus targeted surrounding spans. No gold labels were inferred from retrieval rankings.",\n    "scope": "Draft dev set; no acoustic listening or independent human review. Speech quotes are checked transcript readings; terminology is corroborated against visible frames where available.",\n    "visual_gold": "Narrow spans around inspected keyframe times; they do not assume every final reveal was visible throughout the slide interval.",\n    "known_uncertainty": [\n      "hVU5Mw19V_c: the seed heading says type II 1.5 kg but the calculation uses 2/5; preserve this lecturer inconsistency.",\n      "Temporal recall is any positive overlap and may overestimate success for very long production slide intervals. Context recall additionally checks retained text."\n    ],\n    "review_directory": "runs/exp/baseline/review"\n  },\n  "questions": [\n    {\n      "id": "cs_0001_q01",\n      "video": "cs_0001",\n      "question": "What is the title of the talk?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          29,\n          33\n        ]\n      ],\n      "answer": "A Partial Order View of Message-Passing Communication Models",\n      "evidence": "A Partial Order View of Message-Passing Communication Models",\n      "key_terms": [\n        "Partial Order",\n        "Message-Passing"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 31.066667\n    },\n    {\n      "id": "cs_0001_q02",\n      "video": "cs_0001",\n      "question": "Which university and country are printed under the authors?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          29,\n          33\n        ]\n      ],\n      "answer": "Université Côte d’Azur, France.",\n      "evidence": "Université Côte d’Azur, France",\n      "key_terms": [\n        "Université",\n        "France"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 31.066667\n    },\n    {\n      "id": "cs_0001_q03",\n      "video": "cs_0001",\n      "question": "Why does the speaker say he is a little anxious?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          17,\n          34\n        ]\n      ],\n      "answer": "This is his first conference.",\n      "evidence": "a little anxious because this is my first conference ever",\n      "key_terms": [\n        "first conference"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 31.066667\n    },\n    {\n      "id": "cs_0001_q04",\n      "video": "cs_0001",\n      "question": "Why does asynchronous communication make verification difficult?",\n      "type": "both",\n      "gold_spans": [\n        [\n          137,\n          166\n        ]\n      ],\n      "answer": "It introduces an infinite state space and undecidability.",\n      "evidence": "Asynchronous communication ⇔ Infinite state space (undecidability); those properties ... become undecidable",\n      "key_terms": [\n        "infinite state space",\n        "undecidable"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00001.jpg",\n      "keyframe_time_sec": 162.966667\n    },\n    {\n      "id": "cs_0001_q05",\n      "video": "cs_0001",\n      "question": "How many communication models does the speaker compare?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          401,\n          428\n        ]\n      ],\n      "answer": "Seven.",\n      "evidence": "any of these seven communication models",\n      "key_terms": [\n        "seven communication models"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 426.266667\n    },\n    {\n      "id": "cs_0001_q06",\n      "video": "cs_0001",\n      "question": "Which FIFO model is also called the mailbox model?",\n      "type": "term",\n      "gold_spans": [\n        [\n          424,\n          428\n        ]\n      ],\n      "answer": "FIFO n-1.",\n      "evidence": "FIFO n-1 or mailbox",\n      "key_terms": [\n        "FIFO",\n        "mailbox"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 426.266667\n    },\n    {\n      "id": "cs_0001_q07",\n      "video": "cs_0001",\n      "question": "What are the three process labels in the MSC drawing?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          424,\n          428\n        ]\n      ],\n      "answer": "p, q, r.",\n      "evidence": "Three vertical process lines labelled p, q, r, joined by message arrows.",\n      "key_terms": [\n        "p",\n        "q",\n        "r"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 426.266667\n    },\n    {\n      "id": "cs_0001_q08",\n      "video": "cs_0001",\n      "question": "In the upper model-checking Venn diagram, where is the bounded STW set?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          799,\n          802\n        ]\n      ],\n      "answer": "Inside the intersection of system behaviours and the property language.",\n      "evidence": "Bounded STW L_com(S) is drawn inside both L_com(S) and L(phi).",\n      "key_terms": [\n        "Bounded",\n        "STW"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00011.jpg",\n      "keyframe_time_sec": 801.366667\n    },\n    {\n      "id": "cs_0001_q09",\n      "video": "cs_0001",\n      "question": "What does k = 1 mean for weakly k-synchronous MSCs?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          878,\n          933\n        ]\n      ],\n      "answer": "Purely synchronous communication.",\n      "evidence": "when k equals 1, we are purely synchronous communication",\n      "key_terms": [\n        "purely synchronous"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00013.jpg",\n      "keyframe_time_sec": 903.466667\n    },\n    {\n      "id": "cs_0001_q10",\n      "video": "cs_0001",\n      "question": "Is weakly synchronous model-checking decidable for causally ordered communication in the table?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          901,\n          905\n        ]\n      ],\n      "answer": "It is marked undecidable.",\n      "evidence": "Causally ordered / Weakly sync: red cross; legend × = undecidable",\n      "key_terms": [\n        "Causally ordered",\n        "Weakly sync",\n        "undecidable"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00013.jpg",\n      "keyframe_time_sec": 903.466667\n    },\n    {\n      "id": "cs_0001_q11",\n      "video": "cs_0001",\n      "question": "What can a user do with the speaker’s web tool?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1044,\n          1080\n        ]\n      ],\n      "answer": "Draw MSCs and check realizability under the considered asynchronous models.",\n      "evidence": "draw your own MSCs and see if they can be realizable with any of the asynchronous communication models",\n      "key_terms": [\n        "draw your own MSCs",\n        "realizable"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 1069.2\n    },\n    {\n      "id": "cs_0001_q12",\n      "video": "cs_0001",\n      "question": "What Python package implements the full verification tool, and how do I install it?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Not specified in this lecture.",\n      "evidence": "The lecture proposes a verification tool and links an MSC analyser; no Python installation procedure is given.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "cs_0001/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 1069.2\n    },\n    {\n      "id": "D_kuulxKuxc_q01",\n      "video": "D_kuulxKuxc",\n      "question": "10 tru 3 bang may?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          78.4,\n          102.4\n        ]\n      ],\n      "answer": "7.",\n      "evidence": "10 -3 bằng 7",\n      "key_terms": [\n        "7"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 92.36\n    },\n    {\n      "id": "D_kuulxKuxc_q02",\n      "video": "D_kuulxKuxc",\n      "question": "Trong thử thách đầu tiên, 10 trừ 6 bằng bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          78.4,\n          102.4\n        ]\n      ],\n      "answer": "4.",\n      "evidence": "10 -6 bằng 4",\n      "key_terms": [\n        "4"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 92.36\n    },\n    {\n      "id": "D_kuulxKuxc_q03",\n      "video": "D_kuulxKuxc",\n      "question": "Trên đường đi của bạn gấu, 14 được tách thành 10 cộng số nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          226.6,\n          230.6\n        ]\n      ],\n      "answer": "4.",\n      "evidence": "14 = 10 + ?",\n      "key_terms": [\n        "14",\n        "10"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 228.6\n    },\n    {\n      "id": "D_kuulxKuxc_q04",\n      "video": "D_kuulxKuxc",\n      "question": "Hình 12 chấm tròn gạch bỏ 3 chấm biểu diễn phép tính nào?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          402.68,\n          406.68\n        ]\n      ],\n      "answer": "12 − 3 = 9.",\n      "evidence": "12 − 3 = ?; ba chấm cuối bị gạch, còn chín chấm.",\n      "key_terms": [\n        "12",\n        "3",\n        "9"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00026.jpg",\n      "keyframe_time_sec": 404.68\n    },\n    {\n      "id": "D_kuulxKuxc_q05",\n      "video": "D_kuulxKuxc",\n      "question": "Bạn voi tính 13 trừ 4 được kết quả gì?",\n      "type": "both",\n      "gold_spans": [\n        [\n          694.52,\n          720.64\n        ]\n      ],\n      "answer": "9.",\n      "evidence": "13 trừ 3 bằng 10, 10 trừ 1 bằng 9; Vậy 13 − 4 = 9",\n      "key_terms": [\n        "13",\n        "4",\n        "9"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00044.jpg",\n      "keyframe_time_sec": 719.52\n    },\n    {\n      "id": "D_kuulxKuxc_q06",\n      "video": "D_kuulxKuxc",\n      "question": "Bạn voi bớt 4 chấm tròn theo hai bước nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          694.52,\n          720.64\n        ]\n      ],\n      "answer": "Bớt 3 chấm ngoài khay trước rồi bớt 1 chấm trong khay.",\n      "evidence": "bớt 3 chấm tròn ngoài khay trước, rồi bớt 1 chấm tròn ... trong khay",\n      "key_terms": [\n        "3 chấm tròn",\n        "1 chấm tròn"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00044.jpg",\n      "keyframe_time_sec": 719.52\n    },\n    {\n      "id": "D_kuulxKuxc_q07",\n      "video": "D_kuulxKuxc",\n      "question": "“Đếm bớt” để tính 11 trừ 5 bắt đầu từ số nào?",\n      "type": "term",\n      "gold_spans": [\n        [\n          1079.64,\n          1106.12\n        ]\n      ],\n      "answer": "Bắt đầu từ 11 rồi đếm lùi 5 lần.",\n      "evidence": "Em ... bớt 5 bắt đầu từ 11 ... 10, 9, 8 ... 7 ... 6",\n      "key_terms": [\n        "bắt đầu từ 11"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00060.jpg",\n      "keyframe_time_sec": 1104.64\n    },\n    {\n      "id": "D_kuulxKuxc_q08",\n      "video": "D_kuulxKuxc",\n      "question": "Kết quả 11 − 5 được viết trên màn hình là gì?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1102.64,\n          1106.12\n        ]\n      ],\n      "answer": "6.",\n      "evidence": "11 − 5 = 6",\n      "key_terms": [\n        "11",\n        "5",\n        "6"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00060.jpg",\n      "keyframe_time_sec": 1104.64\n    },\n    {\n      "id": "D_kuulxKuxc_q09",\n      "video": "D_kuulxKuxc",\n      "question": "Nhà bà có 18 quả gấc, 9 quả đã chín, còn mấy quả chưa chín?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1789.04,\n          1824.2\n        ]\n      ],\n      "answer": "9 quả.",\n      "evidence": "có 18 quả, trong đó có 9 quả đã chín, còn lại 18 -9 bằng 9 ... chưa chín",\n      "key_terms": [\n        "18",\n        "9",\n        "chưa chín"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00098.jpg",\n      "keyframe_time_sec": 1794.2\n    },\n    {\n      "id": "D_kuulxKuxc_q10",\n      "video": "D_kuulxKuxc",\n      "question": "Cô nhờ ai quay lại các tình huống dùng phép trừ của học sinh?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1789.04,\n          1824.2\n        ]\n      ],\n      "answer": "Cha mẹ.",\n      "evidence": "Hãy nhờ cha mẹ quay lại và gửi về",\n      "key_terms": [\n        "cha mẹ",\n        "quay lại"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00098.jpg",\n      "keyframe_time_sec": 1794.2\n    },\n    {\n      "id": "D_kuulxKuxc_q11",\n      "video": "D_kuulxKuxc",\n      "question": "Sau vòng tròn màu xanh trên đường đi của gấu là phép trừ nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          226.6,\n          230.6\n        ]\n      ],\n      "answer": "17 − 7.",\n      "evidence": "The blue circle on the top right connects to 17 − 7.",\n      "key_terms": [\n        "17",\n        "7"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 228.6\n    },\n    {\n      "id": "D_kuulxKuxc_q12",\n      "video": "D_kuulxKuxc",\n      "question": "Cô hướng dẫn giải phương trình bậc hai bằng công thức nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Bài này không hướng dẫn phương trình bậc hai.",\n      "evidence": "Bài học tập trung vào phép trừ qua 10.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "D_kuulxKuxc/01_slides/keyframes/seg_00098.jpg",\n      "keyframe_time_sec": 1794.2\n    },\n    {\n      "id": "hVU5Mw19V_c_q01",\n      "video": "hVU5Mw19V_c",\n      "question": "Bảng lớp 10 ghi bao nhiêu nam và bao nhiêu nữ?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          486.96,\n          490.96\n        ]\n      ],\n      "answer": "50 nam và 50 nữ.",\n      "evidence": "L10: Nam 50, Nữ 50",\n      "key_terms": [\n        "L10",\n        "50"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 488.96\n    },\n    {\n      "id": "hVU5Mw19V_c_q02",\n      "video": "hVU5Mw19V_c",\n      "question": "Tổng học sinh dùng làm mẫu số khi chọn ngẫu nhiên trong ba lớp là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          463.96,\n          518.96\n        ]\n      ],\n      "answer": "370, không phải 330 như OCR.",\n      "evidence": "Thì cái ... 370 ... 100 trên 370 ... 120 trên 370 ... 150 trên 370; the board shows denominator 370.",\n      "key_terms": [\n        "370"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 488.96\n    },\n    {\n      "id": "hVU5Mw19V_c_q03",\n      "video": "hVU5Mw19V_c",\n      "question": "Trên bảng, xác suất chọn nữ khi đã chọn lớp 12 viết dưới dạng phân số nào?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          486.96,\n          490.96\n        ]\n      ],\n      "answer": "70/150.",\n      "evidence": "P(A|B3) = 70/150",\n      "key_terms": [\n        "70",\n        "150"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00000.jpg",\n      "keyframe_time_sec": 488.96\n    },\n    {\n      "id": "hVU5Mw19V_c_q04",\n      "video": "hVU5Mw19V_c",\n      "question": "Công thức nào trên tiêu đề dùng để tính xác suất ngược sau khi biết biến cố A?",\n      "type": "term",\n      "gold_spans": [\n        [\n          1072.08,\n          1076.08\n        ]\n      ],\n      "answer": "Công thức Bayes.",\n      "evidence": "CÔNG THỨC XÁC SUẤT ĐẦY ĐỦ VÀ CÔNG THỨC BAYES",\n      "key_terms": [\n        "BAYES"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 1074.08\n    },\n    {\n      "id": "hVU5Mw19V_c_q05",\n      "video": "hVU5Mw19V_c",\n      "question": "Khi đã chọn một bạn nữ, thầy yêu cầu tính xác suất bạn đó thuộc khối lớp nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1049.08,\n          1104.08\n        ]\n      ],\n      "answer": "Lớp 12 (B3).",\n      "evidence": "nếu đã nhặt ra một bạn nữ thì ... rơi vào lớp 12 là bao nhiêu ... tức là B3",\n      "key_terms": [\n        "lớp 12",\n        "B3"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 1074.08\n    },\n    {\n      "id": "hVU5Mw19V_c_q06",\n      "video": "hVU5Mw19V_c",\n      "question": "Trong ví dụ hạt giống, ba tỉ lệ nảy mầm được ghi là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1555.08,\n          1559.08\n        ]\n      ],\n      "answer": "Loại I: 90%; II: 80%; III: 60%.",\n      "evidence": "Loại I: 90% nảy mầm; Loại II: 80% nảy mầm; Loại III: 60% nảy mầm",\n      "key_terms": [\n        "90",\n        "80",\n        "60"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1557.08\n    },\n    {\n      "id": "hVU5Mw19V_c_q07",\n      "video": "hVU5Mw19V_c",\n      "question": "Dòng mô tả loại II trên bảng ghi khối lượng mấy kg?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          1555.08,\n          1559.08\n        ]\n      ],\n      "answer": "1,5 kg. Dòng thay số phía dưới dùng 2/5, nên ví dụ có sự không nhất quán.",\n      "evidence": "Loại II; 1,5 kg; 80% nảy mầm (visible heading, not OCR 4,5 kg).",\n      "key_terms": [\n        "1,5",\n        "kg"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1557.08\n    },\n    {\n      "id": "hVU5Mw19V_c_q08",\n      "video": "hVU5Mw19V_c",\n      "question": "Kết quả cuối thầy đọc cho xác suất hạt đã nảy mầm thuộc nhóm III là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1490,\n          1549\n        ]\n      ],\n      "answer": "46,75%.",\n      "evidence": "số ... 0 .4675, tức là 46 .75 %; the board also shows 46,75%.",\n      "key_terms": [\n        "46.75"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1557.08\n    },\n    {\n      "id": "hVU5Mw19V_c_q09",\n      "video": "hVU5Mw19V_c",\n      "question": "Trong ví dụ viện bỏng, bệnh nhân bỏng nhiệt và bỏng hóa chất chiếm tỉ lệ nào?",\n      "type": "both",\n      "gold_spans": [\n        [\n          1549.64,\n          1587.08\n        ],\n        [\n          1789.28,\n          1793.28\n        ]\n      ],\n      "answer": "70% bỏng nhiệt, 30% bỏng hóa chất.",\n      "evidence": "70 % ... bỏng nhiệt và 30 % ... bỏng do hóa chất; board: Bỏng nhiệt 70%, Bỏng hoá chất 30%.",\n      "key_terms": [\n        "70",\n        "30"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00015.jpg",\n      "keyframe_time_sec": 1791.28\n    },\n    {\n      "id": "hVU5Mw19V_c_q10",\n      "video": "hVU5Mw19V_c",\n      "question": "Xác suất biến chứng chung trong ví dụ viện bỏng bằng bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1720,\n          1787\n        ]\n      ],\n      "answer": "0,36 = 36%.",\n      "evidence": "0,3 × 0,7 + 0,5 × 0,3 = 9/25 = 36%; ... bằng 0,36",\n      "key_terms": [\n        "0,36"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00015.jpg",\n      "keyframe_time_sec": 1791.28\n    },\n    {\n      "id": "hVU5Mw19V_c_q11",\n      "video": "hVU5Mw19V_c",\n      "question": "Khi tính xác suất bỏng nhiệt nếu đã biến chứng, mẫu số của công thức Bayes là gì?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1720,\n          1787\n        ]\n      ],\n      "answer": "P(A), xác suất biến chứng chung, bằng 0,36.",\n      "evidence": "cái mẫu số ở dưới này ... pa ở đây là bằng 0 ,36",\n      "key_terms": [\n        "mẫu số",\n        "0,36"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00015.jpg",\n      "keyframe_time_sec": 1791.28\n    },\n    {\n      "id": "hVU5Mw19V_c_q12",\n      "video": "hVU5Mw19V_c",\n      "question": "Thầy cho biết độ nhạy và độ đặc hiệu của xét nghiệm COVID là bao nhiêu?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không có ví dụ xét nghiệm COVID trong bài.",\n      "evidence": "Các ví dụ được kiểm tra là học sinh, hạt giống và bệnh nhân bỏng.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "hVU5Mw19V_c/01_slides/keyframes/seg_00015.jpg",\n      "keyframe_time_sec": 1791.28\n    },\n    {\n      "id": "L25_V008_q01",\n      "video": "L25_V008",\n      "question": "Công thức định nghĩa xác suất P(A) trên slide là gì?",\n      "type": "term",\n      "gold_spans": [\n        [\n          90.93,\n          94.93\n        ]\n      ],\n      "answer": "P(A) = n(A)/n(Ω).",\n      "evidence": "P(A)=n(A)/n(Ω); n(A) là số kết quả thuận lợi, n(Ω) là số kết quả của phép thử.",\n      "key_terms": [\n        "n(A)",\n        "n(Ω)"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00003.jpg",\n      "keyframe_time_sec": 92.926167\n    },\n    {\n      "id": "L25_V008_q02",\n      "video": "L25_V008",\n      "question": "Hộp ở câu 1 có bao nhiêu quả đỏ và bao nhiêu quả xanh?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          282.52,\n          286.52\n        ]\n      ],\n      "answer": "6 quả đỏ và 9 quả xanh.",\n      "evidence": "15 quả cầu gồm 6 quả màu đỏ ... và 9 quả màu xanh",\n      "key_terms": [\n        "6",\n        "9",\n        "đỏ",\n        "xanh"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 284.517567\n    },\n    {\n      "id": "L25_V008_q03",\n      "video": "L25_V008",\n      "question": "Số phần tử không gian mẫu khi chọn 2 quả từ 15 quả là bao nhiêu?",\n      "type": "both",\n      "gold_spans": [\n        [\n          259.52,\n          300\n        ]\n      ],\n      "answer": "C(15,2) = 105.",\n      "evidence": "Chọn 2 quả cầu từ hộp có 15 quả cầu. Do đó n(Ω)=C(15,2)=105.",\n      "key_terms": [\n        "105"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 284.517567\n    },\n    {\n      "id": "L25_V008_q04",\n      "video": "L25_V008",\n      "question": "Câu 1 có tổng cộng bao nhiêu cách thuận lợi?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          259.52,\n          300\n        ]\n      ],\n      "answer": "27.",\n      "evidence": "15 cộng 12 là bằng 27",\n      "key_terms": [\n        "27"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 284.517567\n    },\n    {\n      "id": "L25_V008_q05",\n      "video": "L25_V008",\n      "question": "Xác suất lấy hai quả khác màu có tổng số chẵn ở câu 1 là gì?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          259.52,\n          300\n        ]\n      ],\n      "answer": "9/35.",\n      "evidence": "27 phần 105 và bằng 9 phần 35; P(A)=9/35",\n      "key_terms": [\n        "9",\n        "35"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 284.517567\n    },\n    {\n      "id": "L25_V008_q06",\n      "video": "L25_V008",\n      "question": "Thầy cộng số cách của hai trường hợp thế nào ở ví dụ quả cầu?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          259.52,\n          300\n        ]\n      ],\n      "answer": "Cộng 15 và 12 để được 27.",\n      "evidence": "lấy 2 trường hợp này ... cộng lại ... 15 cộng 12 là bằng 27",\n      "key_terms": [\n        "15",\n        "12",\n        "27"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 284.517567\n    },\n    {\n      "id": "L25_V008_q07",\n      "video": "L25_V008",\n      "question": "Slide câu 2 chia việc chọn 10 thẻ thành những bước nào?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          456.96,\n          460.96\n        ]\n      ],\n      "answer": "Chọn 1 thẻ chia hết cho 10, 4 thẻ chẵn còn lại, rồi 5 thẻ lẻ.",\n      "evidence": "Bước 1: Chọn 1 thẻ mang số chia hết cho 10; Bước 2: Chọn 4 thẻ mang số chẵn; Bước 3: Chọn 5 thẻ mang số lẻ.",\n      "key_terms": [\n        "1",\n        "4",\n        "5"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 458.9585\n    },\n    {\n      "id": "L25_V008_q08",\n      "video": "L25_V008",\n      "question": "Kết quả xác suất ở câu chọn 10 trong 30 thẻ là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          433.96,\n          464.2\n        ]\n      ],\n      "answer": "99/667.",\n      "evidence": "Ta được kết quả là 99 phần 667",\n      "key_terms": [\n        "99",\n        "667"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 458.9585\n    },\n    {\n      "id": "L25_V008_q09",\n      "video": "L25_V008",\n      "question": "Từ 1 đến 40, ba nhóm số theo số dư chia 3 có bao nhiêu phần tử?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          782.32,\n          786.32\n        ]\n      ],\n      "answer": "Chia hết: 13; dư 1: 14; dư 2: 13.",\n      "evidence": "Số chia cho 3 ... 13 số; số chia cho 3 dư 1 ... 14 số; ... dư 2 ... 13 số.",\n      "key_terms": [\n        "13",\n        "14"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 784.316867\n    },\n    {\n      "id": "L25_V008_q10",\n      "video": "L25_V008",\n      "question": "Xác suất tổng ba số trên thẻ chia hết cho 3 ở câu 3 là gì?",\n      "type": "both",\n      "gold_spans": [\n        [\n          759.32,\n          813.41\n        ]\n      ],\n      "answer": "127/380.",\n      "evidence": "3302 ... chia cho ... C3 của 40 ... 127 phần 380; slide: 127/380",\n      "key_terms": [\n        "127",\n        "380"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 784.316867\n    },\n    {\n      "id": "L25_V008_q11",\n      "video": "L25_V008",\n      "question": "Ví dụ sắp xếp tiếp theo yêu cầu gì với ba học sinh lớp 12?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          759.32,\n          813.41\n        ]\n      ],\n      "answer": "Không có hai học sinh lớp 12 đứng cạnh nhau.",\n      "evidence": "xếp 9 học sinh ... không có 2 em học sinh lớp 12 nào đứng cạnh nhau",\n      "key_terms": [\n        "lớp 12",\n        "đứng cạnh nhau"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 784.316867\n    },\n    {\n      "id": "L25_V008_q12",\n      "video": "L25_V008",\n      "question": "Bài này tính hậu nghiệm liên tục bằng tích phân Bayes như thế nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không trình bày Bayes liên tục.",\n      "evidence": "Bài này giải các bài xác suất đếm hữu hạn.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "L25_V008/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 784.316867\n    },\n    {\n      "id": "L25_V011_q01",\n      "video": "L25_V011",\n      "question": "Tên chuyên đề trên trang vàng là gì?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          10.48,\n          14.48\n        ]\n      ],\n      "answer": "Các tổ chức quốc tế.",\n      "evidence": "CÁC TỔ CHỨC QUỐC TẾ",\n      "key_terms": [\n        "tổ chức quốc tế"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00002.jpg",\n      "keyframe_time_sec": 12.48\n    },\n    {\n      "id": "L25_V011_q02",\n      "video": "L25_V011",\n      "question": "Theo thầy, có bao nhiêu nước đến San Francisco soạn hiến chương LHQ?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          132.2,\n          158.28\n        ]\n      ],\n      "answer": "50 nước.",\n      "evidence": "Đại biểu 50 nước đã đến họp tại San Francisco",\n      "key_terms": [\n        "50 nước"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 157.2\n    },\n    {\n      "id": "L25_V011_q03",\n      "video": "L25_V011",\n      "question": "Hội nghị nào nêu ý tưởng thành lập Liên Hiệp Quốc?",\n      "type": "both",\n      "gold_spans": [\n        [\n          132.2,\n          158.28\n        ]\n      ],\n      "answer": "Hội nghị Yalta, tháng 2 năm 1945.",\n      "evidence": "tại hội nghị ... tháng 2 năm 1945 ... ý tưởng thành lập; slide: Hội nghị Yalta đưa ra ý tưởng thành lập LHQ.",\n      "key_terms": [\n        "1945"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 157.2\n    },\n    {\n      "id": "L25_V011_q04",\n      "video": "L25_V011",\n      "question": "Nguyên tắc nhất trí của LHQ áp dụng giữa nhóm nước nào?",\n      "type": "term",\n      "gold_spans": [\n        [\n          318.2,\n          322.2\n        ]\n      ],\n      "answer": "Năm nước lớn: Anh, Pháp, Mỹ, Liên Xô, Trung Quốc.",\n      "evidence": "Nguyên tắc nhất trí giữa 5 nước lớn (A-P-M-LX-TQ)",\n      "key_terms": [\n        "5",\n        "nước lớn"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 320.2\n    },\n    {\n      "id": "L25_V011_q05",\n      "video": "L25_V011",\n      "question": "Slide ghi hai nhiệm kì Việt Nam là ủy viên không thường trực HĐBA nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          318.2,\n          322.2\n        ]\n      ],\n      "answer": "2008–2009 và 2020–2021.",\n      "evidence": "VN là ủy viên không thường trực ... 2008-2009, 2020-2021",\n      "key_terms": [\n        "2008",\n        "2009",\n        "2020",\n        "2021"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 320.2\n    },\n    {\n      "id": "L25_V011_q06",\n      "video": "L25_V011",\n      "question": "Sơ đồ tư duy Liên Hiệp Quốc có những nhánh nội dung nào?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          318.2,\n          322.2\n        ]\n      ],\n      "answer": "Hoàn cảnh ra đời, mục tiêu, vai trò, nguyên tắc hoạt động, quan hệ với Việt Nam.",\n      "evidence": "Five coloured nodes around LIÊN HIỆP QUỐC: HOÀN CẢNH RA ĐỜI, MỤC TIÊU, VAI TRÒ, NGUYÊN TẮC HOẠT ĐỘNG, QUAN HỆ VỚI VIỆT NAM.",\n      "key_terms": [\n        "MỤC TIÊU",\n        "VAI TRÒ",\n        "NGUYÊN TẮC"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 320.2\n    },\n    {\n      "id": "L25_V011_q07",\n      "video": "L25_V011",\n      "question": "Mốc kí hiệp ước thân thiện và hợp tác của ASEAN trên sơ đồ là năm nào?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          673.2,\n          677.2\n        ]\n      ],\n      "answer": "1976.",\n      "evidence": "1976: Kí hiệp ước thân thiện và hợp tác.",\n      "key_terms": [\n        "1976"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 675.2\n    },\n    {\n      "id": "L25_V011_q08",\n      "video": "L25_V011",\n      "question": "Vai trò của ASEAN được thầy và sơ đồ mô tả ra sao?",\n      "type": "both",\n      "gold_spans": [\n        [\n          650.2,\n          692.64\n        ]\n      ],\n      "answer": "Tổ chức hợp tác kinh tế, chính trị, văn hóa của các quốc gia Đông Nam Á.",\n      "evidence": "tổ chức hợp tác kinh tế chính trị văn hóa ... Đông Nam Á",\n      "key_terms": [\n        "kinh tế",\n        "chính trị",\n        "văn hóa",\n        "Đông Nam Á"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 675.2\n    },\n    {\n      "id": "L25_V011_q09",\n      "video": "L25_V011",\n      "question": "Sau hiệp ước nào tổ chức châu Âu đổi tên thành Liên minh châu Âu?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          870,\n          874\n        ]\n      ],\n      "answer": "Hiệp ước Maastricht.",\n      "evidence": "Sau hiệp ước Masstricht thì đổi tên thành Liên minh châu Âu. (Visible slide misspells Maastricht.)",\n      "key_terms": [\n        "Maastricht"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 872.0,\n      "review_note": "Visible slide spells Masstricht; reference normalizes the treaty name."\n    },\n    {\n      "id": "L25_V011_q10",\n      "video": "L25_V011",\n      "question": "Thầy đề nghị cách ôn tập nào sau khi học ba tổ chức quốc tế?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          847,\n          891.6\n        ]\n      ],\n      "answer": "So sánh để nhận ra điểm khác nhau và tương đồng.",\n      "evidence": "nên tiến hành so sánh để nhận ra điểm khác nhau giữa ba tổ chức quốc tế này",\n      "key_terms": [\n        "so sánh",\n        "khác nhau"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 872.0\n    },\n    {\n      "id": "L25_V011_q11",\n      "video": "L25_V011",\n      "question": "Thầy giới thiệu tổ chức nào có quan hệ mật thiết và Việt Nam là thành viên?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          295.2,\n          325\n        ]\n      ],\n      "answer": "ASEAN.",\n      "evidence": "Việt Nam là một trong những thành viên ... Hiệp hội các quốc gia ở Đông Nam Á ... ASEAN",\n      "key_terms": [\n        "ASEAN",\n        "Việt Nam"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 320.2\n    },\n    {\n      "id": "L25_V011_q12",\n      "video": "L25_V011",\n      "question": "Năm 2026 ASEAN có GDP và dân số chính xác là bao nhiêu?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Bài không cung cấp số liệu năm 2026.",\n      "evidence": "Bài ôn lịch sử các tổ chức quốc tế, không cập nhật GDP/dân số 2026.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "L25_V011/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 872.0\n    },\n    {\n      "id": "L25_V037_q01",\n      "video": "L25_V037",\n      "question": "Trang tiêu đề cho biết chuyên đề cách mạng bao trùm giai đoạn nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          22.52,\n          26.52\n        ]\n      ],\n      "answer": "1930–1945.",\n      "evidence": "CÁC CAO TRÀO CÁCH MẠNG ... (1930–1945)",\n      "key_terms": [\n        "1930",\n        "1945"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00002.jpg",\n      "keyframe_time_sec": 24.52\n    },\n    {\n      "id": "L25_V037_q02",\n      "video": "L25_V037",\n      "question": "Slide ghi phong trào 1930–1931 đạt đỉnh cao vào tháng nào?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          296.2,\n          300.2\n        ]\n      ],\n      "answer": "Tháng 9 năm 1930.",\n      "evidence": "Tháng 9/1930: Phong trào đạt đến đỉnh cao với hình thức vũ trang",\n      "key_terms": [\n        "9",\n        "1930"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 298.2\n    },\n    {\n      "id": "L25_V037_q03",\n      "video": "L25_V037",\n      "question": "Chính quyền được lập ở cấp xã trong cao trào 1930–1931 gọi là gì?",\n      "type": "term",\n      "gold_spans": [\n        [\n          296.2,\n          300.2\n        ]\n      ],\n      "answer": "Xô viết.",\n      "evidence": "Các cấp ủy Đảng tiến hành lập Xô Viết",\n      "key_terms": [\n        "Xô Viết"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 298.2\n    },\n    {\n      "id": "L25_V037_q04",\n      "video": "L25_V037",\n      "question": "Cuộc khủng hoảng kinh tế thế giới nêu ở hoàn cảnh lịch sử kéo dài năm nào?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          185,\n          187.9\n        ]\n      ],\n      "answer": "1929–1933.",\n      "evidence": "Khủng hoảng kinh tế thế giới 1929-1933",\n      "key_terms": [\n        "1929",\n        "1933"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 187.0\n    },\n    {\n      "id": "L25_V037_q05",\n      "video": "L25_V037",\n      "question": "Sau tổn thất, Đảng phục hồi tổ chức và cơ sở quần chúng trong giai đoạn nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          162,\n          188\n        ]\n      ],\n      "answer": "1932–1935.",\n      "evidence": "Từ năm 1932 đến năm 1935 ... phục hồi tổ chức đảng và các cơ sở quân chúng",\n      "key_terms": [\n        "1932",\n        "1935",\n        "phục hồi"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 187.0\n    },\n    {\n      "id": "L25_V037_q06",\n      "video": "L25_V037",\n      "question": "Cao trào 1930–1931 hình thành liên minh nào?",\n      "type": "both",\n      "gold_spans": [\n        [\n          273.2,\n          328.2\n        ]\n      ],\n      "answer": "Liên minh công nông.",\n      "evidence": "đã hình thành nên liên minh công nông; slide: Hình thành liên minh công-nông",\n      "key_terms": [\n        "liên minh công nông"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 298.2\n    },\n    {\n      "id": "L25_V037_q07",\n      "video": "L25_V037",\n      "question": "Vì sao thầy gọi cao trào 1930–1931 là cuộc tập dượt đầu tiên?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          273.2,\n          328.2\n        ]\n      ],\n      "answer": "Phong trào rộng khắp cả nước và có nhiều hình thức đấu tranh triệt để.",\n      "evidence": "lần đầu tiên ... quy mô rộng lớn khắp cả nước, với nhiều hình thức",\n      "key_terms": [\n        "khắp cả nước",\n        "nhiều hình thức"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 298.2\n    },\n    {\n      "id": "L25_V037_q08",\n      "video": "L25_V037",\n      "question": "Bốn nhánh quanh cao trào 1936–1939 trên sơ đồ là gì?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          656.2,\n          660.2\n        ]\n      ],\n      "answer": "Hoàn cảnh lịch sử, chủ trương của Đảng, diễn biến, ý nghĩa lịch sử.",\n      "evidence": "Nodes: HOÀN CẢNH LỊCH SỬ, CHỦ TRƯƠNG CỦA ĐẢNG, DIỄN BIẾN, Ý NGHĨA LỊCH SỬ",\n      "key_terms": [\n        "DIỄN BIẾN",\n        "Ý NGHĨA"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 658.2\n    },\n    {\n      "id": "L25_V037_q09",\n      "video": "L25_V037",\n      "question": "Chủ trương 1936–1939 đấu tranh đòi những quyền nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          656.2,\n          660.2\n        ]\n      ],\n      "answer": "Tự do, dân chủ, cơm áo, hòa bình.",\n      "evidence": "Đòi tự do, dân chủ, cơm áo, hòa bình.",\n      "key_terms": [\n        "tự do",\n        "dân chủ",\n        "cơm áo"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00007.jpg",\n      "keyframe_time_sec": 658.2\n    },\n    {\n      "id": "L25_V037_q10",\n      "video": "L25_V037",\n      "question": "Chỉ thị của Trung ương khi Nhật đảo chính Pháp có tên gì?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          819.8,\n          823.8\n        ]\n      ],\n      "answer": "Nhật–Pháp bắn nhau và hành động của chúng ta.",\n      "evidence": "Trung ương Đảng ra chỉ thị “Nhật-Pháp bắn nhau và hành động của chúng ta”.",\n      "key_terms": [\n        "Nhật",\n        "Pháp",\n        "bắn nhau"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 821.8\n    },\n    {\n      "id": "L25_V037_q11",\n      "video": "L25_V037",\n      "question": "Cuối bài, thầy nhắc cao trào kháng Nhật cứu nước diễn ra tháng năm nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          819,\n          851.8\n        ]\n      ],\n      "answer": "Tháng 3 năm 1945.",\n      "evidence": "cao chào kháng nhật ... diễn ra vào tháng 3 năm 1945",\n      "key_terms": [\n        "3",\n        "1945"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 821.8\n    },\n    {\n      "id": "L25_V037_q12",\n      "video": "L25_V037",\n      "question": "Hiệp định Genève năm 1954 được bài này phân tích từng điều khoản thế nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không phân tích hiệp định Genève.",\n      "evidence": "Phạm vi chuyên đề là các cao trào 1930–1945.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "L25_V037/01_slides/keyframes/seg_00008.jpg",\n      "keyframe_time_sec": 821.8\n    },\n    {\n      "id": "tHE1mC_n0-E_q01",\n      "video": "tHE1mC_n0-E",\n      "question": "Thầy giới thiệu mình thuộc hệ thống giáo dục nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          46.24,\n          57.44\n        ]\n      ],\n      "answer": "Hệ thống giáo dục Học Mãi.",\n      "evidence": "Thầy là Nguyễn Thành Nam ... thuộc hệ thống giáo dục học mãi",\n      "key_terms": [\n        "học mãi"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00002.jpg",\n      "keyframe_time_sec": 49.6\n    },\n    {\n      "id": "tHE1mC_n0-E_q02",\n      "video": "tHE1mC_n0-E",\n      "question": "Đại lượng đặc trưng cho độ bền vững của hạt nhân là gì?",\n      "type": "both",\n      "gold_spans": [\n        [\n          625,\n          660\n        ],\n        [\n          675.96,\n          679.96\n        ]\n      ],\n      "answer": "Năng lượng liên kết riêng.",\n      "evidence": "độ bền vững của hạt nhân là năng lượng liên kết riêng",\n      "key_terms": [\n        "năng lượng liên kết riêng"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00042.jpg",\n      "keyframe_time_sec": 677.96\n    },\n    {\n      "id": "tHE1mC_n0-E_q03",\n      "video": "tHE1mC_n0-E",\n      "question": "Nhiệt hóa hơi riêng tính từ Q và m bằng công thức nào?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          675.96,\n          679.96\n        ]\n      ],\n      "answer": "L = Q/m.",\n      "evidence": "Câu 10: đáp án A, Q/m.",\n      "key_terms": [\n        "Q",\n        "m"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00042.jpg",\n      "keyframe_time_sec": 677.96\n    },\n    {\n      "id": "tHE1mC_n0-E_q04",\n      "video": "tHE1mC_n0-E",\n      "question": "Đồng chuyển từ rắn sang lỏng để đúc gọi là quá trình gì?",\n      "type": "term",\n      "gold_spans": [\n        [\n          660.96,\n          683.44\n        ]\n      ],\n      "answer": "Nóng chảy.",\n      "evidence": "chuyển từ thể rắn thành thể lỏng ... nóng chảy",\n      "key_terms": [\n        "nóng chảy"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00042.jpg",\n      "keyframe_time_sec": 677.96\n    },\n    {\n      "id": "tHE1mC_n0-E_q05",\n      "video": "tHE1mC_n0-E",\n      "question": "Câu 18 cho đơn vị SI của nhiệt nóng chảy riêng là gì?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          914.64,\n          918.64\n        ]\n      ],\n      "answer": "J/kg.",\n      "evidence": "D. J/kg",\n      "key_terms": [\n        "J",\n        "kg"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00065.jpg",\n      "keyframe_time_sec": 916.64\n    },\n    {\n      "id": "tHE1mC_n0-E_q06",\n      "video": "tHE1mC_n0-E",\n      "question": "Ở thể lỏng, phân tử dao động quanh vị trí cân bằng loại nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          914.64,\n          918.64\n        ]\n      ],\n      "answer": "Vị trí cân bằng không cố định (phương án C).",\n      "evidence": "C. dao động quanh vị trí cân bằng không cố định.",\n      "key_terms": [\n        "không cố định"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00065.jpg",\n      "keyframe_time_sec": 916.64\n    },\n    {\n      "id": "tHE1mC_n0-E_q07",\n      "video": "tHE1mC_n0-E",\n      "question": "Khi dòng điện chạy trong dây ở thí nghiệm nam châm đặt trên cân, số chỉ cân tăng hay giảm?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1260,\n          1305\n        ]\n      ],\n      "answer": "Giảm vì nam châm bị lực từ hút lên.",\n      "evidence": "bị hút lên bởi một lực ... lực từ ... cân ... giá trị ... nhỏ hơn",\n      "key_terms": [\n        "lực từ",\n        "nhỏ hơn"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00097.jpg",\n      "keyframe_time_sec": 1280.36\n    },\n    {\n      "id": "tHE1mC_n0-E_q08",\n      "video": "tHE1mC_n0-E",\n      "question": "Trong sơ đồ dữ kiện khung dây, cảm ứng từ đặt theo hướng nào so với mặt phẳng dây?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          1819.64,\n          1823.64\n        ]\n      ],\n      "answer": "Vuông góc với mặt phẳng khung dây.",\n      "evidence": "cảm ứng từ B vuông góc với mặt phẳng khung dây",\n      "key_terms": [\n        "vuông góc",\n        "khung dây"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00134.jpg",\n      "keyframe_time_sec": 1821.64\n    },\n    {\n      "id": "tHE1mC_n0-E_q09",\n      "video": "tHE1mC_n0-E",\n      "question": "Ở bài khung dây, B tăng từ bao nhiêu đến bao nhiêu trong 0,750 giây?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1819.64,\n          1823.64\n        ]\n      ],\n      "answer": "Từ 0,100 T đến 0,600 T.",\n      "evidence": "Trong 0,750 s ... tăng đều từ 0,100 T đến 0,600 T.",\n      "key_terms": [\n        "0,100",\n        "0,600"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00134.jpg",\n      "keyframe_time_sec": 1821.64\n    },\n    {\n      "id": "tHE1mC_n0-E_q10",\n      "video": "tHE1mC_n0-E",\n      "question": "Chu kỳ bán rã của radium trong lớp dạ quang đồng hồ là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          2284.24,\n          2296\n        ]\n      ],\n      "answer": "1600 năm.",\n      "evidence": "Có chu kỳ bán rã 1 .600 năm",\n      "key_terms": [\n        "1.600",\n        "năm"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00181.jpg",\n      "keyframe_time_sec": 2290.84\n    },\n    {\n      "id": "tHE1mC_n0-E_q11",\n      "video": "tHE1mC_n0-E",\n      "question": "Đề nêu thu hồi oxy trong trạm vũ trụ từ khí nào do nhà du hành thở ra?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          1819.64,\n          1823.64\n        ]\n      ],\n      "answer": "Carbon dioxide CO2.",\n      "evidence": "thu hồi oxygen O2 từ khí carbon dioxide CO2 do các nhà du hành thở ra",\n      "key_terms": [\n        "CO2"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00134.jpg",\n      "keyframe_time_sec": 1821.64\n    },\n    {\n      "id": "tHE1mC_n0-E_q12",\n      "video": "tHE1mC_n0-E",\n      "question": "Bài giảng cho công thức thiết kế máy gia tốc LHC với bán kính cụ thể nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không trình bày thiết kế máy gia tốc LHC.",\n      "evidence": "Nội dung được kiểm tra là chữa đề thi, không phải thiết kế LHC.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "tHE1mC_n0-E/01_slides/keyframes/seg_00181.jpg",\n      "keyframe_time_sec": 2290.84\n    },\n    {\n      "id": "v2YTz8ToEag_q01",\n      "video": "v2YTz8ToEag",\n      "question": "Trong tình huống hai vợ chồng, người vợ vừa được thưởng bao nhiêu tiền?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          387.98,\n          395.1\n        ]\n      ],\n      "answer": "100 triệu đồng.",\n      "evidence": "em vừa ... thưởng 100 triệu đồng",\n      "key_terms": [\n        "100 triệu"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 389.052519\n    },\n    {\n      "id": "v2YTz8ToEag_q02",\n      "video": "v2YTz8ToEag",\n      "question": "Người chồng lo điều gì khi đầu tư chứng khoán?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          417.13,\n          433.28\n        ]\n      ],\n      "answer": "Rủi ro, có thể mất hết tiền.",\n      "evidence": "anh thấy rủi ro lắm ... Có thể mất hết",\n      "key_terms": [\n        "mất hết"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 432.352829\n    },\n    {\n      "id": "v2YTz8ToEag_q03",\n      "video": "v2YTz8ToEag",\n      "question": "Kinh tế vi mô tập trung vào hai tác nhân nào?",\n      "type": "both",\n      "gold_spans": [\n        [\n          790.32,\n          845.32\n        ]\n      ],\n      "answer": "Người tiêu dùng và doanh nghiệp.",\n      "evidence": "2 cái tác nhân chính ... và doanh nghiệp; slide: người tiêu dùng và hãng sản xuất kinh doanh",\n      "key_terms": [\n        "doanh nghiệp"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 815.323414\n    },\n    {\n      "id": "v2YTz8ToEag_q04",\n      "video": "v2YTz8ToEag",\n      "question": "Sơ đồ khái niệm kinh tế học liệt kê bốn tác nhân nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          813.32,\n          817.32\n        ]\n      ],\n      "answer": "Người tiêu dùng, doanh nghiệp, chính phủ, yếu tố nước ngoài.",\n      "evidence": "Người tiêu dùng; Doanh nghiệp; Chính phủ; Yếu tố nước ngoài.",\n      "key_terms": [\n        "Chính phủ",\n        "nước ngoài"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 815.323414\n    },\n    {\n      "id": "v2YTz8ToEag_q05",\n      "video": "v2YTz8ToEag",\n      "question": "Chi phi co hoi la gi theo dong dinh nghia tren slide?",\n      "type": "term",\n      "gold_spans": [\n        [\n          1153.45,\n          1157.45\n        ]\n      ],\n      "answer": "Giá trị phương án tốt nhất bị bỏ qua khi lựa chọn kinh tế.",\n      "evidence": "Giá trị của phương án tốt nhất bị bỏ qua khi thực hiện sự lựa chọn kinh tế.",\n      "key_terms": [\n        "phương án tốt nhất",\n        "bị bỏ qua"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00024.jpg",\n      "keyframe_time_sec": 1155.452702\n    },\n    {\n      "id": "v2YTz8ToEag_q06",\n      "video": "v2YTz8ToEag",\n      "question": "Chi phí cơ hội giúp quyết định đầu tư theo mức chấp nhận rủi ro như thế nào?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          1466.12,\n          1516\n        ]\n      ],\n      "answer": "Phải chấp nhận đánh đổi; lựa chọn phụ thuộc mức chấp nhận rủi ro của mỗi người.",\n      "evidence": "phân tích ... chúng ta phải chấp nhận sự đánh đổi ... tùy thuộc ... đối tượng chấp nhận rủi ro",\n      "key_terms": [\n        "đánh đổi",\n        "chấp nhận rủi ro"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00024.jpg",\n      "keyframe_time_sec": 1155.452702\n    },\n    {\n      "id": "v2YTz8ToEag_q07",\n      "video": "v2YTz8ToEag",\n      "question": "PPF viết đầy đủ bằng tiếng Anh là gì?",\n      "type": "term",\n      "gold_spans": [\n        [\n          1640,\n          1680\n        ]\n      ],\n      "answer": "Production Possibility Frontier.",\n      "evidence": "Tiếng Anh là Production Possibility Frontier; slide caption ends PPF (ASR says PPS).",\n      "key_terms": [\n        "Production Possibility Frontier"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00026.jpg",\n      "keyframe_time_sec": 1696.400566\n    },\n    {\n      "id": "v2YTz8ToEag_q08",\n      "video": "v2YTz8ToEag",\n      "question": "Điểm G nằm bên trong hay bên ngoài đường PPF?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          1694.4,\n          1698.4\n        ]\n      ],\n      "answer": "G nằm trong đường PPF.",\n      "evidence": "G is drawn inside the frontier in the inspected diagram.",\n      "key_terms": [\n        "G",\n        "PPF"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00026.jpg",\n      "keyframe_time_sec": 1696.400566\n    },\n    {\n      "id": "v2YTz8ToEag_q09",\n      "video": "v2YTz8ToEag",\n      "question": "Tọa độ điểm B trên đồ thị PPF đầu tiên là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1694.4,\n          1698.4\n        ]\n      ],\n      "answer": "X = 11, Y = 27.",\n      "evidence": "Dashed projections from B meet X=11 and Y=27.",\n      "key_terms": [\n        "11",\n        "27"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00026.jpg",\n      "keyframe_time_sec": 1696.400566\n    },\n    {\n      "id": "v2YTz8ToEag_q10",\n      "video": "v2YTz8ToEag",\n      "question": "Giá trị chi phí cơ hội được xác định bằng đại lượng nào của đường PPF?",\n      "type": "both",\n      "gold_spans": [\n        [\n          1897.54,\n          1952.3\n        ]\n      ],\n      "answer": "Giá trị tuyệt đối độ dốc của PPF.",\n      "evidence": "giá trị tuyệt đối ... dốc của đường PPF; slide: |ΔY/ΔX| = |độ dốc đường PPF|",\n      "key_terms": [\n        "PPF",\n        "dốc"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00029.jpg",\n      "keyframe_time_sec": 1922.541407\n    },\n    {\n      "id": "v2YTz8ToEag_q11",\n      "video": "v2YTz8ToEag",\n      "question": "Ba vấn đề cơ bản được đánh dấu chọn ở câu trắc nghiệm là gì?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          2466.08,\n          2470.08\n        ]\n      ],\n      "answer": "Sản xuất như thế nào, cho ai, cái gì.",\n      "evidence": "Checked boxes: 2. Sản xuất như thế nào? 3. Sản xuất cho ai? 4. Sản xuất cái gì?",\n      "key_terms": [\n        "như thế nào",\n        "cho ai",\n        "cái gì"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00038.jpg",\n      "keyframe_time_sec": 2468.079409\n    },\n    {\n      "id": "v2YTz8ToEag_q12",\n      "video": "v2YTz8ToEag",\n      "question": "Giá cổ phiếu nào sẽ tăng ngày mai và nên mua mã nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Bài không dự báo giá hay khuyến nghị mã cổ phiếu.",\n      "evidence": "Bài giải thích lựa chọn, rủi ro và chi phí cơ hội.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "v2YTz8ToEag/01_slides/keyframes/seg_00038.jpg",\n      "keyframe_time_sec": 2468.079409\n    },\n    {\n      "id": "y1CvB950VlU_q01",\n      "video": "y1CvB950VlU",\n      "question": "Để viết chuỗi nhiều dòng trong Python, ví dụ dùng kiểu dấu nháy nào?",\n      "type": "code",\n      "gold_spans": [\n        [\n          110.27,\n          114.27\n        ]\n      ],\n      "answer": "Ba dấu nháy kép hoặc ba dấu nháy đơn.",\n      "evidence": "The editor shows triple double quotes around a3 and triple single quotes around a4.",\n      "key_terms": [\n        "a3",\n        "a4"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 112.266667\n    },\n    {\n      "id": "y1CvB950VlU_q02",\n      "video": "y1CvB950VlU",\n      "question": "Thầy đưa loại nội dung nào vào ví dụ chuỗi nhiều dòng a4?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          87.27,\n          126.97\n        ]\n      ],\n      "answer": "Một câu thơ.",\n      "evidence": "có thể là chèn 1 câu thơ vào ... mình tự chèn 1 câu thơ",\n      "key_terms": [\n        "câu thơ"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00005.jpg",\n      "keyframe_time_sec": 112.266667\n    },\n    {\n      "id": "y1CvB950VlU_q03",\n      "video": "y1CvB950VlU",\n      "question": "Hàm len dùng để kiểm tra gì ở một chuỗi?",\n      "type": "term",\n      "gold_spans": [\n        [\n          205,\n          250\n        ]\n      ],\n      "answer": "Độ dài, số ký tự của chuỗi.",\n      "evidence": "string ... kiểm tra cái chiều dài ... Độ dài ... bao nhiêu ký tự",\n      "key_terms": [\n        "ký tự"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 255.233333\n    },\n    {\n      "id": "y1CvB950VlU_q04",\n      "video": "y1CvB950VlU",\n      "question": "Thêm ba dấu cách làm độ dài ví dụ tăng từ 13 lên mấy?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          205,\n          250\n        ]\n      ],\n      "answer": "16.",\n      "evidence": "13 ... 3 dấu cách ... lên 16",\n      "key_terms": [\n        "13",\n        "16"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 255.233333\n    },\n    {\n      "id": "y1CvB950VlU_q05",\n      "video": "y1CvB950VlU",\n      "question": "len có đếm cả khoảng trắng trong chuỗi không?",\n      "type": "both",\n      "gold_spans": [\n        [\n          250.13,\n          256.1\n        ]\n      ],\n      "answer": "Có.",\n      "evidence": "còn ... trắng nó vẫn đếm bình thường; editor shows spaced a2 and output 16.",\n      "key_terms": [\n        "đếm",\n        "bình thường"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00016.jpg",\n      "keyframe_time_sec": 255.233333\n    },\n    {\n      "id": "y1CvB950VlU_q06",\n      "video": "y1CvB950VlU",\n      "question": "Trong vòng lặp for i in a2, lệnh nào in từng ký tự?",\n      "type": "code",\n      "gold_spans": [\n        [\n          341.73,\n          345.73\n        ]\n      ],\n      "answer": "print(i), thụt vào trong thân vòng lặp.",\n      "evidence": "for i in a2:\\n    print(i)",\n      "key_terms": [\n        "for",\n        "i",\n        "print"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00020.jpg",\n      "keyframe_time_sec": 343.733333\n    },\n    {\n      "id": "y1CvB950VlU_q07",\n      "video": "y1CvB950VlU",\n      "question": "Thầy nhắc in biến i hay in cả a2 trong vòng lặp duyệt chuỗi?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          332.13,\n          346.23\n        ]\n      ],\n      "answer": "In i, không phải in cả a2.",\n      "evidence": "print Y ... không phải print A2 ... phần tử Y trong A2",\n      "key_terms": [\n        "print",\n        "A2"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00020.jpg",\n      "keyframe_time_sec": 343.733333\n    },\n    {\n      "id": "y1CvB950VlU_q08",\n      "video": "y1CvB950VlU",\n      "question": "Phép cộng hai chuỗi trong ví dụ a = a1 + a2 làm gì?",\n      "type": "code",\n      "gold_spans": [\n        [\n          492.97,\n          496.5\n        ]\n      ],\n      "answer": "Nối hai chuỗi.",\n      "evidence": "a=a1+a2; the console shows the contents joined.",\n      "key_terms": [\n        "a1",\n        "a2"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00027.jpg",\n      "keyframe_time_sec": 494.966667\n    },\n    {\n      "id": "y1CvB950VlU_q09",\n      "video": "y1CvB950VlU",\n      "question": "Ví dụ nhân dấu trừ với 40 in bao nhiêu dấu trừ?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          575.97,\n          581.43\n        ]\n      ],\n      "answer": "40 dấu trừ.",\n      "evidence": "dấu trừ nhân ... 40 ... xuất 40 cái dấu trừ",\n      "key_terms": [\n        "40",\n        "dấu trừ"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00032.jpg",\n      "keyframe_time_sec": 577.9\n    },\n    {\n      "id": "y1CvB950VlU_q10",\n      "video": "y1CvB950VlU",\n      "question": "Muốn kiểm tra chữ “Ga” có trong a2 thì dùng toán tử nào?",\n      "type": "code",\n      "gold_spans": [\n        [\n          620.23,\n          638.43\n        ]\n      ],\n      "answer": "in, ví dụ \\"Ga\\" in a2.",\n      "evidence": "kiểm tra ... có nằm trong cái A2 không ... G, A ... in; editor shows print(\\"Ga\\"...).",\n      "key_terms": [\n        "in"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00037.jpg",\n      "keyframe_time_sec": 636.466667\n    },\n    {\n      "id": "y1CvB950VlU_q11",\n      "video": "y1CvB950VlU",\n      "question": "Kết quả True màu xanh được hiển thị ở vùng nào của IDE?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          702.57,\n          705.57\n        ]\n      ],\n      "answer": "Cửa sổ Run/console phía dưới.",\n      "evidence": "The lower Run panel shows True in blue.",\n      "key_terms": [\n        "True"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00043.jpg",\n      "keyframe_time_sec": 703.566667\n    },\n    {\n      "id": "y1CvB950VlU_q12",\n      "video": "y1CvB950VlU",\n      "question": "Bài này triển khai thuật toán sắp xếp merge sort với độ phức tạp nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không triển khai merge sort.",\n      "evidence": "Bài học là chuỗi, len, vòng lặp, nối chuỗi và kiểm tra thành viên.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "y1CvB950VlU/01_slides/keyframes/seg_00049.jpg",\n      "keyframe_time_sec": 749.266667\n    },\n    {\n      "id": "_eDGTHhyQ7M_q01",\n      "video": "_eDGTHhyQ7M",\n      "question": "Các kim loại kiềm có cấu hình electron lớp ngoài cùng dạng nào?",\n      "type": "term",\n      "gold_spans": [\n        [\n          612.38,\n          615.75\n        ]\n      ],\n      "answer": "ns1.",\n      "evidence": "Cấu hình electron ns¹ (slide heading); nhóm IA có số oxi hóa +1.",\n      "key_terms": [\n        "ns¹"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 614.380433\n    },\n    {\n      "id": "_eDGTHhyQ7M_q02",\n      "video": "_eDGTHhyQ7M",\n      "question": "Slide yêu cầu bảo quản kim loại kiềm bằng cách nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          612.38,\n          615.75\n        ]\n      ],\n      "answer": "Ngâm trong dầu hỏa hoặc chất lỏng không phản ứng.",\n      "evidence": "Các kim loại nhóm IA ... bảo quản trong dầu hoả, trong chất lỏng không phản ứng với kim loại.",\n      "key_terms": [\n        "dầu",\n        "hoả"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 614.380433\n    },\n    {\n      "id": "_eDGTHhyQ7M_q03",\n      "video": "_eDGTHhyQ7M",\n      "question": "Tại sao thầy nói các kim loại kiềm đều phản ứng với nước ở điều kiện thường?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          589.38,\n          615.75\n        ]\n      ],\n      "answer": "Thế khử chuẩn của chúng âm hơn thế của nước.",\n      "evidence": "đều âm hơn thế điện cực chuẩn của nước ... đều phản ứng với nước ở điều kiện thường",\n      "key_terms": [\n        "thế",\n        "nước",\n        "điều kiện thường"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00006.jpg",\n      "keyframe_time_sec": 614.380433\n    },\n    {\n      "id": "_eDGTHhyQ7M_q04",\n      "video": "_eDGTHhyQ7M",\n      "question": "Nước muối sinh lý trên slide có nồng độ NaCl bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1081.65,\n          1085.65\n        ]\n      ],\n      "answer": "0,9%.",\n      "evidence": "Nước muối sinh lý (Dung dịch NaCl 0,9%)",\n      "key_terms": [\n        "NaCl",\n        "0,9"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 1083.649233\n    },\n    {\n      "id": "_eDGTHhyQ7M_q05",\n      "video": "_eDGTHhyQ7M",\n      "question": "Ở sơ đồ điện phân NaCl, khí nào thoát ra ở anode?",\n      "type": "visual",\n      "gold_spans": [\n        [\n          1081.65,\n          1085.65\n        ]\n      ],\n      "answer": "Chlorine Cl2.",\n      "evidence": "Anode (+): 2Cl− → Cl2 + 2e; Cl2 arrow above anode.",\n      "key_terms": [\n        "Anode",\n        "Cl2"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00009.jpg",\n      "keyframe_time_sec": 1083.649233\n    },\n    {\n      "id": "_eDGTHhyQ7M_q06",\n      "video": "_eDGTHhyQ7M",\n      "question": "Trong dãy Li, Na, Al, Ca, K, Rb có bao nhiêu kim loại kiềm?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1603.9,\n          1607.9\n        ]\n      ],\n      "answer": "4: Li, Na, K, Rb.",\n      "evidence": "Dãy Li, Na, Al, Ca, K, Rb; Li, Na, K, Rb thuộc nhóm IA, nên có 4. (The slide lists the elements; this answer requires counting.)",\n      "key_terms": [\n        "Li",\n        "Na",\n        "K",\n        "Rb"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1605.9043,\n      "review_note": "ASR says “chọn câu C” although C is 3; gold uses the visibly listed elements and periodic-table membership, not that uncertain ASR choice."\n    },\n    {\n      "id": "_eDGTHhyQ7M_q07",\n      "video": "_eDGTHhyQ7M",\n      "question": "Mạng tinh thể của kim loại kiềm gọi là gì?",\n      "type": "term",\n      "gold_spans": [\n        [\n          1670,\n          1750\n        ]\n      ],\n      "answer": "Lập phương tâm khối.",\n      "evidence": "lập phương tâm khói ... mạng tinh thể ... lập phương tâm khối",\n      "key_terms": [\n        "lập phương tâm"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1605.9043\n    },\n    {\n      "id": "_eDGTHhyQ7M_q08",\n      "video": "_eDGTHhyQ7M",\n      "question": "Thầy nêu độ đặc khít của mạng lập phương tâm khối là bao nhiêu?",\n      "type": "numeric",\n      "gold_spans": [\n        [\n          1670,\n          1750\n        ]\n      ],\n      "answer": "68%.",\n      "evidence": "độ đặc khích ... 68 %",\n      "key_terms": [\n        "68"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00012.jpg",\n      "keyframe_time_sec": 1605.9043\n    },\n    {\n      "id": "_eDGTHhyQ7M_q09",\n      "video": "_eDGTHhyQ7M",\n      "question": "Kim loại kiềm có độ cứng thấp vì hai nguyên nhân nào ở câu 6?",\n      "type": "both",\n      "gold_spans": [\n        [\n          1884.01,\n          1909.91\n        ]\n      ],\n      "answer": "Mạng lập phương tâm khối kém đặc khít và lực liên kết kim loại yếu.",\n      "evidence": "ý số 2 không phải ... Lực liên kết kim loại yếu ... chọn ý 1 và 3; slide circles A: (1) và (3).",\n      "key_terms": [\n        "1",\n        "3"\n      ],\n      "evidence_source": "either",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00015.jpg",\n      "keyframe_time_sec": 1909.0071\n    },\n    {\n      "id": "_eDGTHhyQ7M_q10",\n      "video": "_eDGTHhyQ7M",\n      "question": "Trong câu Solvay, bảng học sinh ghi X và T là những chất nào?",\n      "type": "slide",\n      "gold_spans": [\n        [\n          2351.12,\n          2355.12\n        ]\n      ],\n      "answer": "X = NaHCO3; T = CO2.",\n      "evidence": "Table: X NaHCO3; T CO2",\n      "key_terms": [\n        "NaHCO3",\n        "CO2"\n      ],\n      "evidence_source": "ocr",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00018.jpg",\n      "keyframe_time_sec": 2353.117433\n    },\n    {\n      "id": "_eDGTHhyQ7M_q11",\n      "video": "_eDGTHhyQ7M",\n      "question": "Thầy sửa hai chất Y và Z trong bảng Solvay thành gì?",\n      "type": "speech",\n      "gold_spans": [\n        [\n          2210,\n          2330\n        ]\n      ],\n      "answer": "Y = NH4Cl, Z = NH3.",\n      "evidence": "Y này phải là NH4Cl Z này phải là NH3",\n      "key_terms": [\n        "NH4Cl",\n        "NH3"\n      ],\n      "evidence_source": "speech",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00018.jpg",\n      "keyframe_time_sec": 2353.117433\n    },\n    {\n      "id": "_eDGTHhyQ7M_q12",\n      "video": "_eDGTHhyQ7M",\n      "question": "Bài hướng dẫn quy trình điện phân nhôm công nghiệp và nhiệt độ vận hành nào?",\n      "type": "unanswerable",\n      "gold_spans": [],\n      "answer": "Không hướng dẫn điện phân nhôm.",\n      "evidence": "Bài tập trung vào nhóm IA, NaCl và quy trình Solvay.",\n      "key_terms": [],\n      "evidence_source": "none",\n      "checked_keyframe": "_eDGTHhyQ7M/01_slides/keyframes/seg_00018.jpg",\n      "keyframe_time_sec": 2353.117433\n    }\n  ]\n}\n', 'pipeline/shared/source_paths.py': '"""Locate source files embedded verbatim by notebook builders and offline tools."""\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nLEGACY_GOLD = ROOT / "runs/legacy-gold" / "kaggle" / "lecture_rag" / "gold"\nMODULE_PATHS = {\n    "common.py": "shared/common.py",\n    "production_runtime.py": "shared/production_runtime.py",\n    "production_queue.py": "shared/production_queue.py",\n    "detection.py": "stages/s01_segment/detection.py",\n    "adfilter.py": "stages/s02_gate/adfilter.py",\n    "visual.py": "stages/s03_ocr/visual.py",\n    "transcription.py": "stages/s04_asr/transcription.py",\n    "alignment.py": "stages/s05_align/alignment.py",\n    "textclean.py": "stages/s05_align/textclean.py",\n    "indexing.py": "stages/s06_index/indexing.py",\n    "retrieval.py": "stages/s06_index/retrieval.py",\n    "demo_answer.py": "demo/demo_answer.py",\n    "demo_attribution.py": "demo/demo_attribution.py",\n    "demo_capture.py": "demo/demo_capture.py",\n    "demo_chat.py": "demo/demo_chat.py",\n    "demo_evidence.py": "demo/demo_evidence.py",\n    "demo_runtime.py": "demo/demo_runtime.py",\n    "demo_sources.py": "demo/demo_sources.py",\n}\n\n\ndef module_path(filename):\n    return ROOT / MODULE_PATHS[filename]\n', 'backend/app/features/search/retrieval.py': '"""Evidence search over a video\'s stage-05 index (pipeline), for the Q&A endpoint.\n\nSame method as pipeline/stages/s06_index/retrieval.py, without PyTorch: the multilingual-E5 query encoder\nruns as ONNX (onnxruntime + tokenizers, pinned to the revision the index was built with), the\nkeyword side is the index\'s SQLite FTS5 table. Meaning counts more than keywords (weighted\nreciprocal rank fusion) because ASR spelling errors and common words make BM25 noisy.\n\nEvidence is grouped per slide: its title, cleaned OCR text, lecture speech, time span and\nkeyframe image, so the answer model can also read numbers from the slide image itself.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport sqlite3\nimport threading\nimport unicodedata\nfrom contextlib import closing\nfrom dataclasses import asdict, dataclass, field\nfrom functools import lru_cache\nfrom pathlib import Path\n\nfrom app.shared import storage\nfrom .rerank import rerank\n\nINDEX_DIR_NAME = "index"\nE5_REPO = "intfloat/multilingual-e5-small"\nE5_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"\nSEMANTIC_WEIGHT, LEXICAL_WEIGHT, RRF_K = 1.0, 0.5, 60\nCANDIDATES, MAX_SLIDES, MAX_IMAGES = 20, 3, 2\nRERANK = os.environ.get("LECTURE_RERANK", "off").strip().lower() in {"1", "true", "on", "yes"}\nRERANK_CANDIDATES = 30\n# Legacy boost retained for ablation; disabled after query-time OCR grouping (eval/PHASE2.md).\nSOURCE_BOOST = {"ocr": 1.05, "region": 1.05}\n# Independent query-time ablations; no index files are changed.\nRANKED_CONTEXT = True\nNEIGHBOR_EXPANSION = True\nVIETNAMESE_LEXICAL = True\nLEXICAL_PHRASES = True\nOCR_DEDUP = True\nSLIDE_DIVERSITY = True\nAPPLY_SOURCE_BOOST = False\nMAX_CONTEXT_CHARS = 8000\nNEIGHBOR_GAP_SEC = 2.0\n\n\n@dataclass(frozen=True)\nclass RetrievalOptions:\n    ranked_context: bool = RANKED_CONTEXT\n    neighbors: bool = NEIGHBOR_EXPANSION\n    vietnamese_lexical: bool = VIETNAMESE_LEXICAL\n    phrases: bool = LEXICAL_PHRASES\n    dedup: bool = OCR_DEDUP\n    diversity: bool = SLIDE_DIVERSITY\n    source_boost: bool = APPLY_SOURCE_BOOST\n\n\ndef retrieval_options(features=None):\n    if features is None:\n        return RetrievalOptions()\n    return RetrievalOptions(**{key: (key in features) if key != "source_boost" else "no_boost" not in features\n                               for key in asdict(RetrievalOptions())})\n\n\n# A follow-up usually asks about the slides just cited: their chunks get this extra factor.\nFOLLOW_UP_BOOST = 1.1\nSTOPWORDS = set("""a ai anh ay ba bang bao bi boi cac cach cai can cho chi chua chung co con cua cung da dang\nde den deu di do doi duoc gi gio ha hay hon khi khong la lai lam len luc ma minh mot moi muon nao nay\nneu ngay nhieu nhu nhung noi o phai qua ra rang rat roi sao se sau tai the thi thoi theo thu tren tro\ntrong tu va van ve vi voi vua xin what how why which when who is are of an to in on for and or do does""".split())\nVI_STOPWORDS = set("""ai anh ấy bằng bao bị bởi các cái cần cho chưa chúng có còn của cũng đã đang\nđể đến đều đi đó được gì giờ hãy hơn khi không là lại làm mà mình một mỗi muốn nào này\nnếu nhiều như những nói ở phải quá ra rằng rất rồi sao sẽ sau tại thì thôi theo trên\ntrong và vẫn về vì với vừa xin what how why which when who is are of an to in on for and or do does""".split())\nTEMPORAL = ("slide nay", "doan nay", "luc nay", "cho nay", "hien tai", "this slide", "current slide")\n\n\ndef normalize_search(text: str) -> str:\n    """Identical to pipeline/shared/common.py: the FTS5 index was built with it."""\n    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))\n    return "".join(char for char in text if not unicodedata.combining(char))\n\n\ndef index_dir(video_id: str) -> Path:\n    return storage.DATA_DIR / INDEX_DIR_NAME / video_id\n\n\ndef has_index(video_id: str) -> bool:\n    if storage.cloud():\n        from app.shared import supabase as supa\n        return supa.has_index(video_id)\n    folder = index_dir(video_id)\n    return all((folder / name).exists() for name in ("chunks.jsonl", "embeddings.npy", "lexical.sqlite3"))\n\n\n# Query encoders by the embedding model named in an index\'s manifest (stage 05 or reembed_index.py).\nENCODERS = {\n    "intfloat/multilingual-e5-small": {"revision": E5_REVISION, "files": ("onnx/model.onnx",),\n                                       "tokenizer": "onnx/tokenizer.json", "prefix": "query: ", "pooling": "mean"},\n    "BAAI/bge-m3": {"revision": "5617a9f61b028005a4858fdac845db406aefb181",\n                    "files": ("onnx/model.onnx", "onnx/model.onnx_data", "onnx/Constant_7_attr__value"),\n                    "tokenizer": "onnx/tokenizer.json", "prefix": "", "pooling": "cls"},\n}\n\n\nclass OnnxEncoder:\n    """Text encoder in ONNX (no PyTorch), normalized like sentence-transformers: E5 = mean pooling\n    with the "query: " prefix, bge-m3 = CLS pooling (its dense vector) without prefix."""\n\n    def __init__(self, repo: str = E5_REPO, max_tokens: int = 512):\n        import onnxruntime\n        from huggingface_hub import hf_hub_download\n        from tokenizers import Tokenizer\n\n        spec = ENCODERS[repo]\n        paths = [hf_hub_download(repo, name, revision=spec["revision"]) for name in spec["files"]]\n        tokenizer = hf_hub_download(repo, spec["tokenizer"], revision=spec["revision"])\n        self.session = onnxruntime.InferenceSession(paths[0], providers=["CPUExecutionProvider"])\n        self.tokenizer = Tokenizer.from_file(tokenizer)\n        self.tokenizer.enable_truncation(max_tokens)\n        self.prefix, self.pooling = spec["prefix"], spec["pooling"]\n        self.inputs = {item.name for item in self.session.get_inputs()}\n        self.outputs = [item.name for item in self.session.get_outputs()]\n        self.lock = threading.Lock()\n\n    def encode_many(self, texts: list, prefix: str | None = None):\n        """L2-normalized vectors of texts (padded batch); prefix defaults to the query prefix."""\n        import numpy as np\n\n        prefix = self.prefix if prefix is None else prefix\n        encodings = [self.tokenizer.encode(prefix + text) for text in texts]\n        width = max(len(e.ids) for e in encodings)\n        ids = np.zeros((len(texts), width), dtype=np.int64)\n        mask = np.zeros_like(ids)\n        pad = self.tokenizer.token_to_id("<pad>") or 0\n        ids[:] = pad\n        for row, encoding in enumerate(encodings):\n            ids[row, :len(encoding.ids)] = encoding.ids\n            mask[row, :len(encoding.ids)] = 1\n        feed = {"input_ids": ids, "attention_mask": mask}\n        if "token_type_ids" in self.inputs:\n            feed["token_type_ids"] = np.zeros_like(ids)\n        with self.lock:\n            outputs = dict(zip(self.outputs, self.session.run(None, feed)))\n        hidden = outputs.get("last_hidden_state", outputs.get("token_embeddings", next(iter(outputs.values()))))\n        if "sentence_embedding" in outputs:        # bge-m3\'s export pools (CLS) itself\n            pooled = outputs["sentence_embedding"]\n        elif self.pooling == "cls":\n            pooled = hidden[:, 0]\n        else:\n            pooled = (hidden * mask[..., None]).sum(1) / mask.sum(1, keepdims=True)\n        return pooled / (np.linalg.norm(pooled, axis=1, keepdims=True) + 1e-12)\n\n    def encode(self, text: str):\n        return self.encode_many([text])[0]\n\n\nOnnxE5 = OnnxEncoder   # the old name, E5 by default\n\n\n@lru_cache(maxsize=2)\ndef default_encoder(repo: str = E5_REPO) -> OnnxEncoder:\n    return OnnxEncoder(repo)\n\n\n@dataclass\nclass LectureIndex:\n    folder: Path\n    chunks: list\n    vectors: object\n    segments: list\n    by_segment: dict = field(default_factory=dict)\n    model: str = E5_REPO\n    video_id: str = ""   # set for Supabase indexes (folder is None)\n\n\n@lru_cache(maxsize=16)\ndef load_index(video_id: str) -> LectureIndex:\n    import numpy as np\n\n    folder = index_dir(video_id)\n    if storage.cloud():\n        from app.shared import supabase as supa\n        chunks, vectors, segments = supa.load_index(video_id)\n        return LectureIndex(None, chunks, vectors, segments, {s["segment_id"]: s for s in segments}, E5_REPO,\n                            video_id)\n    chunks = [json.loads(line) for line in (folder / "chunks.jsonl").read_text(encoding="utf-8").splitlines() if line]\n    vectors = np.load(folder / "embeddings.npy", allow_pickle=False)\n    if len(vectors) != len(chunks):\n        raise ValueError("Chỉ mục hỏng: số vector khác số đoạn.")\n    segments_file = folder / "lecture" / "segments.json"\n    segments = json.loads(segments_file.read_text(encoding="utf-8")) if segments_file.exists() else []\n    manifest = folder / "manifest.json"\n    model = E5_REPO\n    if manifest.exists():\n        model = json.loads(manifest.read_text(encoding="utf-8")).get("parameters", {}).get("embedding_model", E5_REPO)\n    return LectureIndex(folder, chunks, vectors, segments, {s["segment_id"]: s for s in segments},\n                        model if model in ENCODERS else E5_REPO)\n\n\ndef lexical_query(question, vietnamese=True, phrases=False):\n    # Remove only unambiguous accented function words; bare syllables may be technical terms.\n    original = re.findall(r"[^\\W_]+", unicodedata.normalize("NFC", question.casefold()))\n    words = [normalize_search(word) for word in original]\n    keep = [word not in VI_STOPWORDS if vietnamese else folded not in STOPWORDS\n            for word, folded in zip(original, words)]\n    terms = list(dict.fromkeys(word for word, allowed in zip(words, keep) if allowed))[:64]\n    if not vietnamese and not terms:\n        terms = list(dict.fromkeys(words))[:64]\n    pairs = list(dict.fromkeys((words[i], words[i + 1]) for i in range(len(words) - 1)\n                              if keep[i] and keep[i + 1] and words[i] in terms and words[i + 1] in terms))[:32] if phrases else []\n    return terms, pairs\n\n\ndef lexical_hits(folder: Path | None, question: str, count: int, video_id: str = "",\n                 options: RetrievalOptions | None = None) -> list[str]:\n    options = options or RetrievalOptions()\n    terms, phrases = lexical_query(question, options.vietnamese_lexical, options.phrases)\n    if not terms:\n        return []\n    if folder is None:   # Supabase: the same terms against the chunks\' unaccented tsvector\n        from app.shared import supabase as supa\n        return supa.lexical_hits(video_id, terms, count, phrases=phrases)\n    expression = " OR ".join(f\'"{term}"\' for term in terms)\n    if phrases:\n        expression += " OR " + " OR ".join(f\'"{a} {b}"\' for a, b in phrases)\n    with closing(sqlite3.connect((folder / "lexical.sqlite3").resolve().as_uri() + "?mode=ro", uri=True)) as db:\n        return [row[0] for row in db.execute(\n            "SELECT chunk_id FROM evidence WHERE evidence MATCH ? ORDER BY bm25(evidence) LIMIT ?",\n            (expression, count))]\n\n\n# Matched with diacritics: without them "đó" = "độ", "nó" = "nổ", "còn" = "con".\nFOLLOW_UP_WORDS = {"nó", "đó", "kia", "ấy", "vậy", "còn", "thêm", "nữa", "it", "that", "those", "them", "they"}\nFOLLOW_UP_STARTS = ("còn ", "thế còn ", "vậy ", "thế thì ", "tại sao vậy", "what about", "and ", "how about", "why is that")\n\n\ndef is_follow_up(question: str) -> bool:\n    """A short question that leans on the previous one ("còn cái kia thì sao?", "nó là gì?")."""\n    text = unicodedata.normalize("NFC", question).lower().strip()\n    words = re.findall(r"[^\\W_]+", text)\n    if not words or len(words) > 10:\n        return False\n    return text.startswith(FOLLOW_UP_STARTS) or bool(FOLLOW_UP_WORDS & set(words))\n\n\ndef search_question(question: str, previous: str | None) -> str:\n    """The text to search with: a follow-up question is searched together with the one before it."""\n    if previous and previous.strip() and is_follow_up(question):\n        return f"{previous.strip()} {question.strip()}"\n    return question\n\n\ndef is_ad(segment: dict) -> bool:\n    """An ad segment (content_kind "ad"), or one the ad filter only flagged ("flag" mode)."""\n    return segment.get("content_kind") == "ad" or bool(segment.get("ad_flagged"))\n\n\ndef segment_at(segments: list, time_sec: float):\n    for index, segment in enumerate(segments):\n        if segment["start_sec"] <= time_sec < segment["end_sec"]:\n            if is_ad(segment):   # paused on an ad: the slide shown before it\n                earlier = [s for s in segments[:index] if not is_ad(s)]\n                return earlier[-1] if earlier else None\n            return segment\n    return None\n\n\ndef content_groups(index: LectureIndex):\n    """Conservative OCR groups; changed numbers are never treated as near duplicates."""\n    if hasattr(index, "_content_groups"):\n        return index._content_groups\n    groups, exact, representatives = {}, {}, []\n    for chunk in index.chunks:\n        cid = chunk["chunk_id"]\n        if chunk["source_type"] != "ocr":\n            groups[cid] = cid\n            continue\n        normalized = normalize_search(chunk["text"])\n        words = re.findall(r"[^\\W_]+", normalized)\n        identity = " ".join(normalized.split())\n        numbers = tuple(re.findall(r"\\d+(?:[.,]\\d+)*|[=+*/−-]", normalized))\n        shingles = {tuple(words[i:i + 3]) for i in range(len(words) - 2)}\n        group = exact.get(identity)\n        if group is None and len(words) >= 16:\n            for other_id, other_numbers, other_shingles in representatives:\n                if numbers == other_numbers and len(shingles & other_shingles) / max(1, len(shingles | other_shingles)) >= 0.9:\n                    group = other_id\n                    break\n        if group is None:\n            group = cid\n            if len(words) >= 16:\n                representatives.append((cid, numbers, shingles))\n        exact[identity] = group\n        groups[cid] = group\n    index._content_groups = groups\n    return groups\n\n\ndef context_candidates(index, sid, ranked, similarities, options, sources=None, question=""):\n    chunks = [c for c in index.chunks if c["segment_id"] == sid\n              and (sources is None or c["source_type"] in sources)]\n    by_id = {c["chunk_id"]: c for c in index.chunks}\n    positions = {c["chunk_id"]: i for i, c in enumerate(index.chunks)}\n    matched = [by_id[cid] for cid in ranked if by_id[cid]["segment_id"] == sid]\n    anchor = matched[0] if matched else (chunks[0] if chunks else None)\n    if anchor is None:\n        return [], []\n    selected, seen = [], set()\n\n    def add(chunk, reason, priority):\n        if chunk["chunk_id"] in seen or (sources is not None and chunk["source_type"] not in sources):\n            return\n        seen.add(chunk["chunk_id"])\n        segment = index.by_segment.get(chunk["segment_id"], {})\n        selected.append({**chunk, "selection_reason": reason, "priority": priority,\n                         "slide_start_sec": segment.get("start_sec", chunk["start_sec"]),\n                         "slide_end_sec": segment.get("end_sec", chunk["end_sec"])})\n\n    add(anchor, "matched", 0)\n    for kind in ("ocr", "speech", "region"):\n        candidates = [c for c in chunks if c["source_type"] == kind]\n        if candidates:\n            best = max(candidates, key=lambda c: float(similarities[positions[c["chunk_id"]]]))\n            add(best, "same_slide", 1)\n        elif kind in {"ocr", "speech"}:\n            segment = index.by_segment.get(sid, {})\n            text = (segment.get("visual", {}).get("ocr_text_clean") if kind == "ocr"\n                    else segment.get("speech", {}).get("text"))\n            if text:\n                add({"chunk_id": sid + ":" + kind, "segment_id": sid, "source_type": kind,\n                     "text": text, "start_sec": segment["start_sec"], "end_sec": segment["end_sec"]},\n                    "same_slide", 1)\n    for chunk in matched[1:]:\n        add(chunk, "matched", 2)\n    speech = sorted((c for c in chunks if c["source_type"] == "speech"), key=lambda c: c["start_sec"])\n    seeds = [c for c in selected if c["source_type"] == "speech"][:2]\n    for seed in seeds:\n        position = next((i for i, c in enumerate(speech) if c["chunk_id"] == seed["chunk_id"]), None)\n        if position is None:\n            continue\n        for n in (position - 1, position + 1):\n            if 0 <= n < len(speech):\n                add(speech[n], "speech_window", 3)\n    if options.neighbors:\n        segment = index.by_segment.get(sid, {})\n        ordered = sorted(index.segments, key=lambda s: s["start_sec"])\n        position = next((i for i, s in enumerate(ordered) if s["segment_id"] == sid), None)\n        explain = any(p in normalize_search(question) for p in\n                      ("giai thich", "cac buoc", "quy trinh", "the nao", "tai sao", "explain", "steps", "how", "why"))\n        for delta in (-1, 1):\n            boundary = segment.get("start_sec" if delta < 0 else "end_sec", anchor["start_sec"])\n            edge = anchor["start_sec" if delta < 0 else "end_sec"]\n            needed = explain or len(anchor["text"]) < 80 or abs(edge - boundary) <= NEIGHBOR_GAP_SEC\n            if position is None or not needed or not 0 <= position + delta < len(ordered):\n                continue\n            neighbor = ordered[position + delta]\n            gap = (segment.get("start_sec", 0) - neighbor["end_sec"] if delta < 0\n                   else neighbor["start_sec"] - segment.get("end_sec", 0))\n            if is_ad(neighbor) or gap > NEIGHBOR_GAP_SEC:\n                continue\n            adjacent = [c for c in index.chunks if c["segment_id"] == neighbor["segment_id"]]\n            narration = sorted((c for c in adjacent if c["source_type"] == "speech"),\n                               key=lambda c: c["start_sec"], reverse=delta < 0)\n            for chunk in narration[:1]:\n                # A visual boundary alone does not justify jumping over a long silent gap.\n                speech_gap = (anchor["start_sec"] - chunk["end_sec"] if delta < 0\n                              else chunk["start_sec"] - anchor["end_sec"])\n                if explain or anchor["source_type"] != "speech" or speech_gap <= NEIGHBOR_GAP_SEC:\n                    add(chunk, "previous" if delta < 0 else "next", 4)\n    return matched, selected\n\n\ndef search(video_id: str, question: str, current_time: float | None = None, encoder=None,\n           sources: frozenset | None = None, prefer_segments: frozenset = frozenset(),\n           options: RetrievalOptions | None = None, *, rerank_scorer=None) -> dict:\n    """Ranked chunks fused per slide -> {\'slides\': [...], \'best_semantic\': float, \'temporal\': bool}.\n    sources limits search and evidence to some chunk types ("speech", "ocr", "region") for ablations.\n    LECTURE_RERANK=on reranks the top 30 RRF candidates before selecting slides.\n    Scored pools add best_rerank and chunk-level scores; unavailable models raise.\n    rerank_scorer injects a logit scorer without loading a model.\n    """\n    import numpy as np\n\n    index = load_index(video_id)\n    options = options or RetrievalOptions()\n    temporal = current_time is not None and any(p in normalize_search(question) for p in TEMPORAL)\n    current = segment_at(index.segments, current_time) if temporal else None\n    vector = (encoder or default_encoder(index.model)).encode(question)\n    similarities = index.vectors @ vector\n    if sources is not None:\n        allowed = np.array([c["source_type"] in sources for c in index.chunks], dtype=bool)\n        similarities = np.where(allowed, similarities, -1.0)\n    by_id = {chunk["chunk_id"]: n for n, chunk in enumerate(index.chunks)}\n    kinds = [c["source_type"] for c in index.chunks]\n    scores: dict[str, float] = {}\n    groups = content_groups(index) if options.dedup else {cid: cid for cid in by_id}\n    representatives = {}\n    # Group before taking the twenty candidates, so copies cannot consume the pool.\n    for n in np.argsort(-similarities, kind="stable"):\n        cid = index.chunks[n]["chunk_id"]\n        if sources is None or kinds[n] in sources:\n            representatives.setdefault(groups[cid], cid)\n    if options.dedup and (prefer_segments or current):\n        def preference(candidate):\n            sid = index.chunks[by_id[candidate]]["segment_id"]\n            return (bool(current and sid == current["segment_id"]), sid in prefer_segments,\n                    float(similarities[by_id[candidate]]))\n\n        for cid in by_id:\n            group = groups[cid]\n            if group not in representatives:\n                continue\n            previous = representatives[group]\n            if preference(cid) > preference(previous):\n                representatives[group] = cid\n    semantic = list(representatives.values())[:CANDIDATES]\n    for rank, cid in enumerate(semantic, 1):\n        scores[cid] = scores.get(cid, 0) + SEMANTIC_WEIGHT / (RRF_K + rank)\n    lexical = lexical_hits(index.folder, question, len(index.chunks) if options.dedup else\n                           (CANDIDATES if sources is None else CANDIDATES * 4), video_id, options)\n    lexical = [cid for cid in lexical if cid in by_id and (sources is None or kinds[by_id[cid]] in sources)]\n    if options.dedup:\n        lexical = list(dict.fromkeys(representatives[groups[cid]] for cid in lexical))\n    for rank, cid in enumerate(lexical[:CANDIDATES], 1):\n        scores[cid] = scores.get(cid, 0) + LEXICAL_WEIGHT / (RRF_K + rank)\n    for cid in scores:\n        if options.source_boost:\n            scores[cid] *= SOURCE_BOOST.get(kinds[by_id[cid]], 1.0)\n        if prefer_segments and any(c["segment_id"] in prefer_segments for c in index.chunks\n                                   if groups[c["chunk_id"]] == groups[cid]):\n            scores[cid] *= FOLLOW_UP_BOOST\n    ranked = sorted(scores, key=lambda cid: -scores[cid])\n    reranked = {}\n    if RERANK:\n        candidates = [index.chunks[by_id[cid]] for cid in ranked[:RERANK_CANDIDATES]]\n        rescored = rerank(question, candidates, scorer=rerank_scorer)\n        reranked = {c["chunk_id"]: c for c in rescored}\n        ranked = list(reranked)\n    order: list[str] = []\n    if temporal:\n        if current:\n            order.append(current["segment_id"])\n    diversity_groups = content_groups(index) if options.diversity else groups\n    used_groups = {diversity_groups[cid] for cid in ranked if index.chunks[by_id[cid]]["segment_id"] in order}\n    for cid in ranked:\n        sid = index.chunks[by_id[cid]]["segment_id"]\n        if options.diversity and diversity_groups[cid] in used_groups and sid not in order:\n            continue\n        if sid not in order:\n            order.append(sid)\n            used_groups.add(diversity_groups[cid])\n        if len(order) >= MAX_SLIDES:\n            break\n    slides = []\n    for rank, sid in enumerate(order):\n        segment = index.by_segment.get(sid, {})\n        chunks = [c for c in index.chunks if c["segment_id"] == sid]\n        visual = segment.get("visual", {})\n        speech = segment.get("speech", {}).get("text") or " ".join(c["text"] for c in chunks if c["source_type"] == "speech")\n        ocr = visual.get("ocr_text_clean") or " ".join(c["text"] for c in chunks if c["source_type"] == "ocr")\n        figures = [c["text"] for c in chunks if c["source_type"] == "region"]\n        if sources is not None:\n            speech = speech if "speech" in sources else ""\n            ocr = ocr if "ocr" in sources else ""\n            figures = figures if "region" in sources else []\n        image = visual.get("image_path") or (chunks[0].get("image_path") if chunks else None)\n        best = max((float(similarities[by_id[c["chunk_id"]]]) for c in chunks), default=0.0)   # -1 = source left out\n        fused = max((scores.get(c["chunk_id"], 0.0) for c in chunks), default=0.0)\n        matched, evidence = context_candidates(index, sid, ranked, similarities, options, sources, question)\n        for chunk in evidence:\n            scored = reranked.get(chunk["chunk_id"])\n            if scored is not None:\n                chunk.update(rerank_score=scored["rerank_score"], rerank_rank=scored["rerank_rank"])\n        if options.dedup:\n            for chunk in evidence:\n                chunk["occurrences"] = [{"chunk_id": c["chunk_id"], "segment_id": c["segment_id"],\n                                         "start_sec": c["start_sec"], "end_sec": c["end_sec"]}\n                                        for c in index.chunks if groups[c["chunk_id"]] == groups.get(chunk["chunk_id"])]\n        slides.append({"segment_id": sid, "rank": rank,\n                       "start_sec": segment.get("start_sec", chunks[0]["start_sec"] if chunks else 0),\n                       "end_sec": segment.get("end_sec", chunks[-1]["end_sec"] if chunks else 0),\n                       "title": visual.get("title_clean", visual.get("title")) or (chunks[0].get("title") if chunks else ""),\n                       "ocr_text": ocr, "speech_text": speech, "figures": figures,\n                       "image": str(index.folder / "lecture" / image) if image and index.folder else None,\n                       "semantic_score": round(best, 4), "score": round(fused, 6),\n                       "video_id": video_id, "matched_chunk_ids": [c["chunk_id"] for c in matched],\n                       "matched_spans": [{"chunk_id": c["chunk_id"], "start_sec": c["start_sec"],\n                                          "end_sec": c["end_sec"]} for c in matched],\n                       "evidence_candidates": evidence})\n    return {"slides": slides, "temporal": temporal,\n            "question": question, "retrieval_options": asdict(options),\n            "best_semantic": round(float(similarities.max()), 4) if len(similarities) else 0.0,\n            **({"best_rerank": next(iter(reranked.values()))["rerank_score"]} if reranked else {})}\n\n\ndef video_title(video_id: str) -> str:\n    meta = storage.load_video_meta(video_id) or {}\n    return Path(meta.get("original_filename") or video_id).stem\n\n\ndef search_many(video_ids: list, question: str, current_video: str | None = None,\n                current_time: float | None = None, max_slides: int = MAX_SLIDES + 1,\n                prefer_segments: frozenset = frozenset(), options: RetrievalOptions | None = None) -> dict:\n    """search() on several indexed videos. RRF scores come from per-video ranks (every video\'s\n    top slide scores about the same), so slides are merged by E5 cosine, which all indexes share.\n    A "this slide" match in the video being watched stays first."""\n    results = [search(vid, question, current_time if vid == current_video else None,\n                      prefer_segments=prefer_segments, options=options)\n               for vid in dict.fromkeys(video_ids) if has_index(vid)]\n    if not results:\n        return {"slides": [], "temporal": False, "best_semantic": 0.0}\n    pinned = [r["slides"][0] for r in results if r["temporal"] and r["slides"]]\n    rest = sorted((s for r in results for s in r["slides"] if s not in pinned), key=lambda s: (-s["semantic_score"], -s["score"]))\n    slides = (pinned + rest)[:max_slides]\n    for slide in slides:\n        slide["video_title"] = video_title(slide["video_id"])\n    found = {"slides": slides, "temporal": bool(pinned),\n             "best_semantic": max(r["best_semantic"] for r in results), "videos": len(results),\n             "question": question, "retrieval_options": asdict(options or RetrievalOptions())}\n    if any("best_rerank" in r for r in results):\n        found["best_rerank"] = max(r["best_rerank"] for r in results if "best_rerank" in r)\n    return found\n\n\ndef clock(seconds: float) -> str:\n    seconds = int(seconds)\n    return f"{seconds // 60:02d}:{seconds % 60:02d}"\n\n\ndef _evidence_source(item: dict, kind: str) -> dict:\n    """Snapshot the cited span, never a deduplicated occurrence or the parent slide span."""\n    chunk_ids = [item["chunk_id"]] if "chunk_id" in item else list(item.get("matched_chunk_ids", []))\n    return {"video_id": item.get("video_id", ""), "t_start": item["start_sec"],\n            "t_end": item["end_sec"], "slide_id": item["segment_id"],\n            "segment_id": item["segment_id"], "chunk_ids": chunk_ids, "kind": kind,\n            "title": item.get("title") or "", "video_title": item.get("video_title", "")}\n\n\ndef _legacy_context(found: dict, speech_chars: int = 2500, ocr_chars: int = 1800,\n                    *, evidence_map: dict | None = None, max_chars: int | None = None) -> str:\n    """Evidence text for the answer model, slide by slide in time order."""\n    blocks = []\n    several = len({slide.get("video_id") for slide in found["slides"]}) > 1\n    for slide in sorted(found["slides"], key=lambda s: (s.get("video_title", ""), s["start_sec"])):\n        where = f"Video {slide.get(\'video_title\', \'\')} · " if several else ""\n        parts = [f"[{where}Slide {slide[\'segment_id\'].split(\':\')[-1]} · {clock(slide[\'start_sec\'])}–{clock(slide[\'end_sec\'])}]"]\n        if slide["title"]:\n            parts.append(f"Tiêu đề: {slide[\'title\']}")\n        if slide["ocr_text"]:\n            note = ("Chữ trên slide (OCR, có thể sai số/dấu):\\n" if evidence_map is not None else\n                    "Chữ trên slide (OCR, có thể sai số/dấu — đối chiếu ảnh nếu có):\\n")\n            parts.append(note + slide["ocr_text"][:ocr_chars])\n        for figure in slide["figures"][:3]:\n            parts.append("Hình/bảng/công thức: " + figure[:600])\n        if slide["speech_text"]:\n            parts.append("Lời giảng:\\n" + slide["speech_text"][:speech_chars])\n        block = "\\n".join(parts)\n        if evidence_map is not None:\n            evidence_id = f"E{len(evidence_map) + 1}"\n            block = f"[{evidence_id}] " + block\n            if max_chars is not None:\n                remaining = max_chars - sum(len(b) for b in blocks) - 2 * len(blocks)\n                # Never send a partial ID/header or map an entirely omitted block.\n                if remaining <= len(block.split("\\n", 1)[0]):\n                    break\n                block = block[:remaining]\n            evidence_map[evidence_id] = _evidence_source(slide, "slide")\n        blocks.append(block)\n    return "\\n\\n".join(blocks)\n\n\ndef evidence_block(chunk, several=False, evidence_id: str | None = None):\n    where = f"Video {chunk.get(\'video_title\', chunk.get(\'video_id\', \'\'))} · " if several else ""\n    label = {"speech": "Lời giảng", "ocr": "Chữ trên slide (OCR)", "region": "Hình/bảng/công thức"}[chunk["source_type"]]\n    return ((f"[{evidence_id}] " if evidence_id else "")\n            + f"[{where}Slide {chunk[\'segment_id\'].split(\':\')[-1]} · "\n            f"{clock(chunk[\'start_sec\'])}–{clock(chunk[\'end_sec\'])}] {label}\\n"\n            + (f"Tiêu đề: {chunk[\'title\']}\\n" if chunk.get("title") else "")\n            + chunk["text"])\n\n\ndef build_context(found: dict, speech_chars: int = 2500, ocr_chars: int = 1800,\n                  max_chars: int = MAX_CONTEXT_CHARS, options: RetrievalOptions | None = None) -> str:\n    """Backward-compatible context without evidence IDs."""\n    return _build_context(found, speech_chars, ocr_chars, max_chars, options)\n\n\ndef build_context_with_ids(found: dict, speech_chars: int = 2500, ocr_chars: int = 1800,\n                           max_chars: int = MAX_CONTEXT_CHARS,\n                           options: RetrievalOptions | None = None) -> tuple[str, dict]:\n    """Assign request-local immutable IDs in final context order, within the text budget."""\n    if max_chars < 1:\n        raise ValueError("Context budget must be positive.")\n    evidence_map = {}\n    context = _build_context(found, speech_chars, ocr_chars, max_chars, options, evidence_map=evidence_map)\n    found["evidence_map"] = evidence_map\n    found["context_chars"] = len(context)\n    return context, evidence_map\n\n\ndef _build_context(found: dict, speech_chars: int, ocr_chars: int, max_chars: int,\n                   options: RetrievalOptions | None, *, evidence_map: dict | None = None) -> str:\n    """Ranked evidence first, then bounded speech windows and conditional neighbors.\n\n    Slide intervals stay in the response for the UI; each text block cites its own span.\n    context_evidence records only text actually sent, for evaluation and debugging.\n    """\n    options = options or RetrievalOptions(**found.get("retrieval_options", {}))\n    if not (options.ranked_context or options.neighbors) or not any("evidence_candidates" in s for s in found["slides"]):\n        return _legacy_context(found, speech_chars, ocr_chars, evidence_map=evidence_map,\n                               max_chars=max_chars if evidence_map is not None else None)\n    if max_chars < 1:\n        raise ValueError("Context budget must be positive.")\n    several = len({s.get("video_id") for s in found["slides"]}) > 1\n    candidates = []\n    for rank, slide in enumerate(found["slides"]):\n        for chunk in slide.get("evidence_candidates", []):\n            if options.ranked_context or chunk["selection_reason"] in {"previous", "next"}:\n                candidates.append({**chunk, "video_id": slide.get("video_id", ""),\n                                   "video_title": slide.get("video_title", ""), "slide_rank": rank})\n    candidates.sort(key=lambda c: (c["priority"], c["slide_rank"]))\n    selected, seen, usage = [], set(), {}\n    legacy = "" if options.ranked_context else _legacy_context(\n        found, speech_chars, ocr_chars, evidence_map=evidence_map,\n        max_chars=max_chars if evidence_map is not None else None)[:max_chars]\n    context = legacy\n    for chunk in candidates:\n        kind = chunk["source_type"]\n        identity = (chunk.get("video_id"), kind, normalize_search(chunk["text"]).strip())\n        if identity in seen:\n            continue\n        key = (chunk.get("video_id"), chunk["segment_id"], kind)\n        limit = speech_chars if kind == "speech" else ocr_chars if kind == "ocr" else 1800\n        if usage.get(key, 0) + len(chunk["text"]) > limit:\n            continue\n        provisional_id = f"E{len(evidence_map) + len(selected) + 1}" if evidence_map is not None else None\n        block = evidence_block(chunk, several, provisional_id)\n        proposed = context + ("\\n\\n" if context else "") + block\n        if len(proposed) > max_chars:\n            continue\n        context = proposed\n        selected.append(chunk)\n        seen.add(identity)\n        usage[key] = usage.get(key, 0) + len(chunk["text"])\n    selected.sort(key=lambda c: (c.get("video_title", ""), c.get("video_id", ""), c["start_sec"], c["end_sec"]))\n    blocks = [legacy] if legacy else []\n    for chunk in selected:\n        evidence_id = None\n        if evidence_map is not None:\n            evidence_id = f"E{len(evidence_map) + 1}"\n            evidence_map[evidence_id] = _evidence_source(chunk, chunk["source_type"])\n            chunk["evidence_id"] = evidence_id\n        blocks.append(evidence_block(chunk, several, evidence_id))\n    context = "\\n\\n".join(blocks)\n    found["context_evidence"] = selected\n    found["context_legacy"] = legacy\n    found["context_chars"] = len(context)\n    return context\n', 'backend/app/features/search/rerank.py': '"""Local-only, lazy BGE cross-encoder scoring. Importing this module loads no model."""\n\nfrom __future__ import annotations\n\nimport importlib.util\nimport math\nimport os\nimport threading\nfrom functools import lru_cache\nfrom pathlib import Path\n\nRERANK_REPO = "BAAI/bge-reranker-v2-m3"\n# TODO A3: fill the reviewed model commit hash before resolving a Hub cache snapshot.\n# Never silently resolve "main". An explicitly provisioned local directory also works.\nRERANK_REVISION = "TODO_FILL_REVIEWED_COMMIT_HASH"\nMAX_TOKENS = 512\nBATCH_SIZE = 8\n_LOAD_LOCK = threading.Lock()\n\n\nclass RerankerUnavailable(RuntimeError):\n    """Installed inference libraries or locally provisioned model files are unavailable."""\n\n\ndef _installed(name: str) -> bool:\n    return importlib.util.find_spec(name) is not None\n\n\ndef _model_directory() -> Path:\n    explicit = os.environ.get("LECTURE_RERANK_MODEL_DIR")\n    if explicit:\n        directory = Path(explicit).expanduser().resolve()\n        if not directory.is_dir():\n            raise RerankerUnavailable("LECTURE_RERANK_MODEL_DIR must be an existing local directory")\n        return directory\n    if len(RERANK_REVISION) != 40 or any(c not in "0123456789abcdef" for c in RERANK_REVISION):\n        raise RerankerUnavailable("Pin RERANK_REVISION or provision LECTURE_RERANK_MODEL_DIR first")\n    from huggingface_hub import snapshot_download\n\n    # This only resolves an existing cache. It must never fetch weights or metadata.\n    return Path(snapshot_download(RERANK_REPO, revision=RERANK_REVISION, local_files_only=True))\n\n\ndef _file(directory: Path, *names: str) -> Path:\n    for name in names:\n        path = directory / name\n        if path.is_file():\n            return path\n    raise FileNotFoundError(f"Missing local reranker file: {\' or \'.join(names)}")\n\n\nclass _OnnxScorer:\n    def __init__(self, directory: Path):\n        import onnxruntime\n        from tokenizers import Tokenizer\n\n        model = _file(directory, "onnx/model.onnx", "model.onnx")\n        tokenizer = _file(directory, "tokenizer.json", "onnx/tokenizer.json")\n        # External ONNX tensor data, if used, must already be beside this model.\n        self.session = onnxruntime.InferenceSession(str(model), providers=["CPUExecutionProvider"])\n        self.tokenizer = Tokenizer.from_file(str(tokenizer))\n        self.tokenizer.enable_truncation(max_length=MAX_TOKENS, strategy="longest_first")\n        self.pad_id = self.tokenizer.token_to_id("<pad>")\n        if self.pad_id is None:\n            self.pad_id = self.tokenizer.token_to_id("[PAD]") or 0\n        self.inputs = {item.name for item in self.session.get_inputs()}\n        self.output = next((item.name for item in self.session.get_outputs() if item.name == "logits"),\n                           self.session.get_outputs()[0].name)\n        self.lock = threading.Lock()\n\n    def __call__(self, pairs: list[tuple[str, str]]) -> list[float]:\n        import numpy as np\n\n        with self.lock:\n            encodings = self.tokenizer.encode_batch(pairs)\n            width = max(len(e.ids) for e in encodings)\n            ids = np.full((len(pairs), width), self.pad_id, dtype=np.int64)\n            mask, types = np.zeros_like(ids), np.zeros_like(ids)\n            for row, encoding in enumerate(encodings):\n                n = len(encoding.ids)\n                ids[row, :n] = encoding.ids\n                mask[row, :n] = encoding.attention_mask\n                types[row, :n] = encoding.type_ids\n            feed = {name: value for name, value in\n                    (("input_ids", ids), ("attention_mask", mask), ("token_type_ids", types))\n                    if name in self.inputs}\n            logits = np.asarray(self.session.run([self.output], feed)[0])\n        if logits.size != len(pairs):\n            raise ValueError("BGE reranker must return one logit per pair")\n        return logits.reshape(-1).tolist()\n\n\nclass _TransformersScorer:\n    def __init__(self, directory: Path):\n        import torch\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n        self.torch = torch\n        self.tokenizer = AutoTokenizer.from_pretrained(str(directory), local_files_only=True,\n                                                       trust_remote_code=False)\n        self.model = AutoModelForSequenceClassification.from_pretrained(\n            str(directory), local_files_only=True, trust_remote_code=False).eval()\n        self.lock = threading.Lock()\n\n    def __call__(self, pairs: list[tuple[str, str]]) -> list[float]:\n        with self.lock, self.torch.inference_mode():\n            inputs = self.tokenizer([q for q, _ in pairs], [text for _, text in pairs],\n                                    padding=True, truncation=True, max_length=MAX_TOKENS,\n                                    return_tensors="pt")\n            logits = self.model(**inputs).logits\n        if logits.numel() != len(pairs):\n            raise ValueError("BGE reranker must return one logit per pair")\n        return logits.reshape(-1).cpu().tolist()\n\n\nclass _SentenceTransformersScorer:\n    def __init__(self, directory: Path):\n        from sentence_transformers import CrossEncoder\n\n        self.model = CrossEncoder(str(directory), max_length=MAX_TOKENS, device="cpu",\n                                  local_files_only=True, trust_remote_code=False)\n        self.lock = threading.Lock()\n\n    def __call__(self, pairs: list[tuple[str, str]]) -> list[float]:\n        from torch.nn import Identity\n\n        # CrossEncoder normally applies a sigmoid; rerank() owns that transformation.\n        with self.lock:\n            values = self.model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False,\n                                        activation_fn=Identity(), convert_to_numpy=True)\n        if values.size != len(pairs):\n            raise ValueError("BGE reranker must return one logit per pair")\n        return values.reshape(-1).tolist()\n\n\n@lru_cache(maxsize=1)\ndef _load_default_scorer():\n    try:\n        directory = _model_directory()\n    except Exception as exc:\n        raise RerankerUnavailable("No locally provisioned BGE reranker; no download attempted") from exc\n    failures = []\n    for modules, implementation in [(("onnxruntime", "tokenizers"), _OnnxScorer),\n                                    (("torch", "transformers"), _TransformersScorer),\n                                    (("torch", "sentence_transformers"), _SentenceTransformersScorer)]:\n        try:\n            if all(_installed(name) for name in modules):\n                return implementation(directory)\n        except Exception as exc:\n            failures.append(exc)\n    error = RerankerUnavailable("No installed backend could load the local BGE reranker")\n    if failures:\n        raise error from failures[-1]\n    raise error\n\n\ndef default_scorer():\n    """Construct once on first nonempty use; failures remain retryable after provisioning."""\n    with _LOAD_LOCK:\n        return _load_default_scorer()\n\n\ndef _sigmoid(logit: float) -> float:\n    # Avoid overflow for confident negative logits. The endpoints may round to 0 or 1.\n    if logit >= 0:\n        return 1.0 / (1.0 + math.exp(-logit))\n    value = math.exp(logit)\n    return value / (1.0 + value)\n\n\ndef rerank(question: str, candidates: list[dict], *, scorer=None,\n           top_k: int | None = None) -> list[dict]:\n    """Copy candidates, batch question/text pairs, sigmoid logits and rank from one.\n\n    Ties retain input order. Injected scorers receive raw text and return logits;\n    the default backend tokenizes each pair jointly with a 512-token total limit.\n    Missing local dependencies/weights raise RerankerUnavailable, never download.\n    """\n    if top_k is not None and (isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 0):\n        raise ValueError("top_k must be a nonnegative integer or None")\n    if not isinstance(question, str):\n        raise TypeError("question must be a string")\n    for candidate in candidates:\n        if "chunk_id" not in candidate or not isinstance(candidate.get("text"), str):\n            raise ValueError("Every candidate needs chunk_id and string text")\n    if not candidates or top_k == 0:\n        return []\n    scorer = default_scorer() if scorer is None else scorer\n    scored = []\n    for start in range(0, len(candidates), BATCH_SIZE):\n        batch = candidates[start:start + BATCH_SIZE]\n        logits = list(scorer([(question, c["text"]) for c in batch]))\n        if len(logits) != len(batch):\n            raise ValueError("Scorer must return one logit per pair")\n        for candidate, value in zip(batch, logits):\n            logit = float(value)\n            if not math.isfinite(logit):\n                raise ValueError("Scorer logits must be finite")\n            scored.append({**candidate, "rerank_score": _sigmoid(logit)})\n    scored.sort(key=lambda c: -c["rerank_score"])\n    for rank, candidate in enumerate(scored, 1):\n        candidate["rerank_rank"] = rank\n    return scored if top_k is None else scored[:top_k]\n'}
import importlib.util, os, sys
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.dont_write_bytecode = True
if USE_GPU and not GPU_RUN_APPROVED:
    raise RuntimeError("Ask the user and obtain approval before running on a GPU")
required = ["numpy", "onnxruntime", "tokenizers", "huggingface_hub"]
if USE_GPU:
    required += ["torch", "transformers"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Missing preinstalled packages: {missing}; no installation attempted")
for path in (INDEX_ROOT, ENCODER_DIR, RERANKER_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Attach the pre-existing local input: {path}")
base = Path("/kaggle/working/rerank-eval-repo")
for name, source in files.items():
    target = base / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
sys.path.insert(0, str(base / "pipeline/eval"))
scorer = None
if USE_GPU:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    if not torch.cuda.is_available():
        raise RuntimeError("Approved GPU run requested but CUDA is unavailable")
    tokenizer = AutoTokenizer.from_pretrained(str(RERANKER_DIR), local_files_only=True,
                                             trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        str(RERANKER_DIR), local_files_only=True, trust_remote_code=False).to("cuda").eval()
    def scorer(pairs):
        inputs = tokenizer([q for q, _ in pairs], [text for _, text in pairs],
                           padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {name: tensor.to("cuda") for name, tensor in inputs.items()}
        with torch.inference_mode():
            logits = model(**inputs).logits
        return logits.reshape(-1).float().cpu().tolist()
    scorer.backend_name = "local-only Transformers CUDA " + torch.cuda.get_device_name(0)
import rerank_eval
sys.argv = ["rerank_eval.py", "--index-root", str(INDEX_ROOT), "--encoder-dir", str(ENCODER_DIR),
            "--reranker-dir", str(RERANKER_DIR), "--name", RUN_NAME, "--output-root", str(OUTPUT_ROOT)]
if QUERY_CACHE is not None:
    sys.argv += ["--query-vectors", str(QUERY_CACHE)]
rerank_eval.main(scorer=scorer)
